### DP2 VIRCAM variance diagnostics

This notebook checks the variance treatment in the DP2 VIRCAM CellCoadd processing before scaling the pipeline to the full ELAIS-S1 field.

The analysis first verifies the resolved pipeline configuration, including whether variance scaling is enabled. The VIRCAM coadd variance plane is then tested empirically against the measured blank-sky noise. This separates single-pixel variance normalization from additional correlated noise introduced by resampling and coaddition.

No variance correction is applied until the existing DP2 products have been validated.

In [1]:
from pathlib import Path

import numpy as np

from lsst.daf.butler import Butler


REPO = Path(
    "../../../dmu4/dmu4_DP2/"
    "dmu4_DP2_Example/data"
)

STEP3B_RUN = "videoStep3b/20261018T154842Z"
STEP3C_RUN = "videoStep3c/20261018T173729Z"

SKYMAP = "lsst_cells_v2"
TRACT = 2877
PATCH = 77

butler = Butler(REPO)

print("Repository :", REPO.resolve())
print("Step3b run:", STEP3B_RUN)
print("Step3c run:", STEP3C_RUN)
print("Test patch :", TRACT, PATCH)

Repository : /local/scratch/es3e23/projects/lsst-ir-fusion/dmu4/dmu4_DP2/dmu4_DP2_Example/data
Step3b run: videoStep3b/20261018T154842Z
Step3c run: videoStep3c/20261018T173729Z
Test patch : 2877 77


### Inspect the resolved variance configuration

The first step is to verify the configuration used by the completed Step3b run. In particular, the detection configuration is checked for `doScaleVariance`, which controls whether the coadd variance plane is rescaled before source detection.

The configuration stored with the run is inspected rather than relying on the current pipeline YAML or package defaults, because these may have changed since the test processing was executed.

In [2]:
config_datasets = []

for dataset_type in butler.registry.queryDatasetTypes():

    name = dataset_type.name

    if "config" not in name.lower():
        continue

    refs = list(
        butler.registry.queryDatasets(
            name,
            collections=STEP3B_RUN,
            findFirst=False,
        )
    )

    if refs:
        config_datasets.append(
            (name, len(refs))
        )


print("Configuration datasets in Step3b")
print("--------------------------------")

for name, nrefs in sorted(config_datasets):
    print(f"{name:45s} {nrefs}")

Configuration datasets in Step3b
--------------------------------
assembleCellCoadd_config                      1
assembleCoadd_config                          1
detectCoaddPeaks_config                       1
rewriteCellCoadd_config                       1
selectDeepCoaddVisits_config                  1


### Check whether variance scaling was enabled

`DetectCoaddSourcesTask` can optionally rescale the coadd variance plane before detection through the `doScaleVariance` configuration option. The stored `detectCoaddPeaks_config` from the completed Step3b run is inspected directly to determine whether this correction was active.

In [3]:
detect_config = butler.get(
    "detectCoaddPeaks_config",
    collections=STEP3B_RUN,
)

print("Config type      :", type(detect_config))
print("doScaleVariance  :", detect_config.doScaleVariance)

Config type      : <class 'lsst.pipe.tasks.multiBand.DetectCoaddSourcesConfig'>
doScaleVariance  : False


### Check variance handling during coadd construction

The detection-stage variance rescaling was disabled in the completed Step3b run. This does not by itself establish how the variance plane was constructed upstream, because variance normalization may also be applied during coadd production.

The stored `assembleCoadd` and `assembleCellCoadd` configurations are therefore searched for settings related to variance or scaling before carrying out an empirical noise test.

In [4]:
def find_variance_settings(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key

            if (
                "variance" in key.lower()
                or "scale" in key.lower()
            ):
                print(f"{path}: {value}")

            find_variance_settings(
                value,
                prefix=path,
            )


for dataset_type in [
    "assembleCoadd_config",
    "assembleCellCoadd_config",
]:

    config = butler.get(
        dataset_type,
        collections=STEP3B_RUN,
    )

    print("\n" + "=" * 80)
    print(dataset_type)
    print("=" * 80)

    find_variance_settings(
        config.toDict()
    )


assembleCoadd_config
select.maxScaledSizeScatter: 0.019
select.maxPsfApCorrSigmaScaledDelta: 0.22
calcErrorFromInputVariance: True
doScaleZeroPoint: False
scaleZeroPoint: {'zeroPoint': 27.0}
assembleStaticSkyModel.select.maxScaledSizeScatter: 0.019
assembleStaticSkyModel.select.maxPsfApCorrSigmaScaledDelta: 0.22
assembleStaticSkyModel.calcErrorFromInputVariance: True
assembleStaticSkyModel.doScaleZeroPoint: False
assembleStaticSkyModel.scaleZeroPoint: {'zeroPoint': 27.0}
doScaleWarpVariance: True
scaleWarpVariance: {'background': {'statisticsProperty': 'MEANCLIP', 'undersampleStyle': 'REDUCE_INTERP_ORDER', 'binSize': 32, 'binSizeX': 0, 'binSizeY': 0, 'algorithm': 'AKIMA_SPLINE', 'ignoredPixelMask': ['DETECTED', 'DETECTED_NEGATIVE', 'BAD', 'SAT', 'NO_DATA', 'INTRP'], 'isNanSafe': False, 'useApprox': False, 'approxOrderX': 6, 'approxOrderY': -1, 'weighting': True, 'doApplyFlatBackgroundRatio': False, 'doFilterSuperPixels': False, 'superPixelFilterSize': 3}, 'maskPlanes': ['DETECTED', 'D

### Variance configuration summary

The completed AP2 run does not apply `doScaleVariance` during coadd-source detection. However, variance scaling is already enabled upstream during coadd construction through `doScaleWarpVariance=True`.

The coadd uncertainties are subsequently calculated from the input variance planes. Therefore, the absence of detection-stage variance scaling does not imply that the final variance plane is unnormalised.

The next step is to test the final VIRCAM CellCoadd variance empirically. A single-pixel blank-sky diagnostic is used first to determine whether the measured pixel fluctuations are consistent with the propagated variance plane. Correlated noise on aperture scales is considered separately later.

In [5]:
BAND = "K"

coadd = butler.get(
    "deep_coadd_cell",
    dataId={
        "skymap": SKYMAP,
        "tract": TRACT,
        "patch": PATCH,
        "band": BAND,
    },
    collections=STEP3B_RUN,
)

In [6]:
print("Type       :", type(coadd))
print("Band       :", coadd.band)
print("BBox       :", coadd.bbox)
print("Backgrounds:", list(coadd.backgrounds.keys()))

Type       : <class 'lsst.images.cells._coadd.CellCoadd'>
Band       : K
BBox       : [y=20850:24150, x=20850:24150]
Backgrounds: ['object']


In [7]:
exp = coadd.to_legacy(copy=True)

print("Legacy type:", type(exp))
print("BBox       :", exp.getBBox())

print("\nMask planes:")
for name, bit in sorted(
    exp.mask.getMaskPlaneDict().items(),
    key=lambda x: x[1],
):
    print(f"{bit:2d}  {name}")

Legacy type: <class 'lsst.afw.image._exposure.ExposureF'>
BBox       : (minimum=(20850, 20850), maximum=(24149, 24149))

Mask planes:
 0  BAD
 1  SAT
 2  INTRP
 3  CR
 4  EDGE
 5  DETECTED
 6  DETECTED_NEGATIVE
 7  SUSPECT
 8  NO_DATA
 9  VIGNETTED
10  STREAK
11  CLIPPED
12  REJECTED
13  INEXACT_PSF


### Inspect the K-band image, variance and masks

The K-band CellCoadd is converted to a legacy `ExposureF` so that its image, variance and mask planes can be inspected directly.

Before defining blank sky, the fraction of pixels affected by each mask plane is measured together with basic statistics of the image and variance arrays. This establishes which masks are relevant and avoids introducing an unnecessarily restrictive blank-sky selection.

In [8]:
image = np.asarray(exp.image.array, dtype=float)
variance = np.asarray(exp.variance.array, dtype=float)
mask = np.asarray(exp.mask.array)

ny, nx = image.shape

print("Array shape:", image.shape)

finite_image = np.isfinite(image)
finite_variance = np.isfinite(variance)
positive_variance = finite_variance & (variance > 0)

print("\nImage / variance summary")
print("------------------------")
print("finite image pixels      :", finite_image.sum())
print("finite variance pixels   :", finite_variance.sum())
print("positive variance pixels :", positive_variance.sum())

print("median image [nJy]       :", np.nanmedian(image))
print(
    "median variance [nJy^2]  :",
    np.nanmedian(variance[positive_variance]),
)
print(
    "median sigma [nJy]       :",
    np.nanmedian(np.sqrt(variance[positive_variance])),
)


print("\nMask-plane occupancy")
print("--------------------")

mask_dict = exp.mask.getMaskPlaneDict()

for name, bit in sorted(
    mask_dict.items(),
    key=lambda item: item[1],
):
    bitmask = 1 << bit
    selected = (mask & bitmask) != 0

    print(
        f"{name:20s}"
        f"{selected.sum():10d}  "
        f"{100 * selected.mean():8.3f}%"
    )

Array shape: (3300, 3300)

Image / variance summary
------------------------
finite image pixels      : 10890000
finite variance pixels   : 10881460
positive variance pixels : 10881460
median image [nJy]       : 1.3216513395309448
median variance [nJy^2]  : 1074.9647216796875
median sigma [nJy]       : 32.786654627754984

Mask-plane occupancy
--------------------
BAD                          0     0.000%
SAT                          0     0.000%
INTRP                        0     0.000%
CR                           0     0.000%
EDGE                     92204     0.847%
DETECTED               1172416    10.766%
DETECTED_NEGATIVE            0     0.000%
SUSPECT                      0     0.000%
NO_DATA                      0     0.000%
VIGNETTED                    0     0.000%
STREAK                       0     0.000%
CLIPPED                 106061     0.974%
REJECTED                709371     6.514%
INEXACT_PSF             804409     7.387%


### Single-pixel variance normalisation

A conservative blank-sky sample is defined using pixels with finite image values, positive finite variance and no active mask bits. This removes detected sources and pixels affected by edges, clipping, rejected inputs or inexact PSF coverage.

For each selected pixel, the image value is divided by the propagated per-pixel uncertainty. The robust scatter of this normalised distribution is measured with the NMAD. If the final variance plane correctly describes the single-pixel noise, the NMAD should be close to unity.

The median of the normalised distribution is removed when measuring the scatter so that a small residual background level does not affect the variance test.

In [9]:
clean = (
    np.isfinite(image)
    & np.isfinite(variance)
    & (variance > 0)
    & (mask == 0)
)

sigma = np.sqrt(variance[clean])
z = image[clean] / sigma

z_med = np.median(z)
z_nmad = 1.4826 * np.median(
    np.abs(z - z_med)
)

# Standard deviation after removing strong outliers.
core = np.abs(z - z_med) < 5.0 * z_nmad
z_std_core = np.std(z[core])

print("K-band single-pixel variance test")
print("---------------------------------")
print("Clean pixels            :", clean.sum())
print("Clean fraction          :", clean.mean())
print("Median image [nJy]      :", np.median(image[clean]))
print("Median sigma [nJy]      :", np.median(sigma))
print("Median image/sigma      :", z_med)
print("NMAD image/sigma        :", z_nmad)
print("Core STD image/sigma    :", z_std_core)
print("Core fraction           :", core.mean())

K-band single-pixel variance test
---------------------------------
Clean pixels            : 8939606
Clean fraction          : 0.8209004591368227
Median image [nJy]      : 0.36503171920776367
Median sigma [nJy]      : 32.78127606648153
Median image/sigma      : 0.01159786357500058
NMAD image/sigma        : 0.3083212738789417
Core STD image/sigma    : 0.3112159006167956
Core fraction           : 0.9999780750963745


### Spatial consistency of the K-band variance

The initial K-band test gives a normalised single-pixel scatter substantially below unity, indicating that the propagated variance is larger than the measured pixel fluctuations.

To determine whether this is a global property of the coadd or is dominated by one region, the same conservative blank-sky test is repeated independently in four image quadrants. For each quadrant, the NMAD of the normalised pixel values and the corresponding implied variance scale factor are reported.

In [10]:
print("K-band spatial variance check")
print("-----------------------------")

y_mid = ny // 2
x_mid = nx // 2

quadrants = {
    "Q1": (slice(0, y_mid), slice(0, x_mid)),
    "Q2": (slice(0, y_mid), slice(x_mid, nx)),
    "Q3": (slice(y_mid, ny), slice(0, x_mid)),
    "Q4": (slice(y_mid, ny), slice(x_mid, nx)),
}

for name, (ys, xs) in quadrants.items():

    q_clean = clean[ys, xs]
    q_image = image[ys, xs][q_clean]
    q_var = variance[ys, xs][q_clean]

    q_z = q_image / np.sqrt(q_var)

    med = np.median(q_z)
    nmad = 1.4826 * np.median(
        np.abs(q_z - med)
    )

    print(f"\n{name}")
    print("  clean pixels          :", len(q_z))
    print("  median image/sigma    :", med)
    print("  NMAD image/sigma      :", nmad)
    print("  implied variance scale:", nmad**2)

K-band spatial variance check
-----------------------------

Q1
  clean pixels          : 2121048
  median image/sigma    : 0.012427769237582376
  NMAD image/sigma      : 0.31404917486061523
  implied variance scale: 0.09862688423063329

Q2
  clean pixels          : 2230509
  median image/sigma    : 0.010697306890589705
  NMAD image/sigma      : 0.31684835789261806
  implied variance scale: 0.10039288189924858

Q3
  clean pixels          : 2228015
  median image/sigma    : 0.011974306749205618
  NMAD image/sigma      : 0.30108883265585407
  implied variance scale: 0.0906544851500649

Q4
  clean pixels          : 2360034
  median image/sigma    : 0.01132735856927828
  NMAD image/sigma      : 0.30246273708266697
  implied variance scale: 0.09148370732353853


### Y-band single-pixel variance test

The same conservative single-pixel test is repeated for the Y band. Using an identical masking and robust-scatter definition allows the inferred variance normalization to be compared directly with the K-band result.

A substantial difference between the Y- and K-band scale factors would indicate that the variance mismatch is band dependent rather than a single global scaling of the VIRCAM coadds.

In [11]:
BAND = "Y"

coadd_y = butler.get(
    "deep_coadd_cell",
    dataId={
        "skymap": SKYMAP,
        "tract": TRACT,
        "patch": PATCH,
        "band": BAND,
    },
    collections=STEP3B_RUN,
)

exp_y = coadd_y.to_legacy(copy=True)

image_y = np.asarray(exp_y.image.array, dtype=float)
variance_y = np.asarray(exp_y.variance.array, dtype=float)
mask_y = np.asarray(exp_y.mask.array)

clean_y = (
    np.isfinite(image_y)
    & np.isfinite(variance_y)
    & (variance_y > 0)
    & (mask_y == 0)
)

sigma_y = np.sqrt(variance_y[clean_y])
z_y = image_y[clean_y] / sigma_y

med_y = np.median(z_y)

nmad_y = 1.4826 * np.median(
    np.abs(z_y - med_y)
)

core_y = np.abs(z_y - med_y) < 5.0 * nmad_y

print("Y-band single-pixel variance test")
print("---------------------------------")
print("Clean pixels            :", clean_y.sum())
print("Clean fraction          :", clean_y.mean())
print("Median image [nJy]      :", np.median(image_y[clean_y]))
print("Median sigma [nJy]      :", np.median(sigma_y))
print("Median image/sigma      :", med_y)
print("NMAD image/sigma        :", nmad_y)
print("Core STD image/sigma    :", np.std(z_y[core_y]))
print("Implied variance scale  :", nmad_y**2)

Y-band single-pixel variance test
---------------------------------
Clean pixels            : 7785494
Clean fraction          : 0.7149213957759413
Median image [nJy]      : 0.12574446201324463
Median sigma [nJy]      : 3.650182661946683
Median image/sigma      : 0.035055631439159934
NMAD image/sigma        : 0.529783843283499
Core STD image/sigma    : 0.5410380307997857
Implied variance scale  : 0.28067092060423504


### J-band single-pixel variance test

The single-pixel variance test is repeated for the J band using the same conservative blank-sky selection and robust-scatter estimator. This provides a third independent measurement of the band-dependent variance normalisation in the DP2 VIRCAM CellCoadds.

In [12]:
BAND = "J"

coadd_j = butler.get(
    "deep_coadd_cell",
    dataId={
        "skymap": SKYMAP,
        "tract": TRACT,
        "patch": PATCH,
        "band": BAND,
    },
    collections=STEP3B_RUN,
)

exp_j = coadd_j.to_legacy(copy=True)

image_j = np.asarray(exp_j.image.array, dtype=float)
variance_j = np.asarray(exp_j.variance.array, dtype=float)
mask_j = np.asarray(exp_j.mask.array)

clean_j = (
    np.isfinite(image_j)
    & np.isfinite(variance_j)
    & (variance_j > 0)
    & (mask_j == 0)
)

sigma_j = np.sqrt(variance_j[clean_j])
z_j = image_j[clean_j] / sigma_j

med_j = np.median(z_j)

nmad_j = 1.4826 * np.median(
    np.abs(z_j - med_j)
)

core_j = np.abs(z_j - med_j) < 5.0 * nmad_j

print("J-band single-pixel variance test")
print("---------------------------------")
print("Clean pixels            :", clean_j.sum())
print("Clean fraction          :", clean_j.mean())
print("Median image [nJy]      :", np.median(image_j[clean_j]))
print("Median sigma [nJy]      :", np.median(sigma_j))
print("Median image/sigma      :", med_j)
print("NMAD image/sigma        :", nmad_j)
print("Core STD image/sigma    :", np.std(z_j[core_j]))
print("Implied variance scale  :", nmad_j**2)

J-band single-pixel variance test
---------------------------------
Clean pixels            : 9125128
Clean fraction          : 0.8379364554637282
Median image [nJy]      : 0.1324881613254547
Median sigma [nJy]      : 8.197535763284723
Median image/sigma      : 0.017400081475583067
NMAD image/sigma        : 0.5186934537107302
Core STD image/sigma    : 0.5250250524594428
Implied variance scale  : 0.2690428989223654


### Interpretation of the single-pixel variance tests

The Y, J and K CellCoadds all show normalised single-pixel scatter substantially below unity when `doScaleVariance=False`. The inferred variance scale factors are approximately 0.28, 0.27 and 0.095 for Y, J and K, respectively.

The effect is strongly band dependent and spatially stable within the K-band test patch. These factors are also consistent with the scale of the empirical VIRCAM variance corrections found previously in DP1.

This indicates that the upstream warp-level variance scaling does not by itself provide the required final coadd-level variance normalisation. The `DetectCoaddSourcesTask` variance-scaling step should therefore be enabled and validated on the test patch before the full DP2 processing is run.

### Verify the scaled Step3b run

The test Step3b processing is repeated with `detectCoaddPeaks:doScaleVariance=True` and written to a separate output collection so that the original products remain unchanged.

The stored configuration confirms that variance scaling is enabled. The available coadd datasets are also inspected to identify bands for which the new processing has completed.

In [13]:
SCALED_STEP3B_RUN = "videoStep3bScaleVariance"

# Check the stored configuration.
scaled_config = butler.get(
    "detectCoaddPeaks_config",
    collections=SCALED_STEP3B_RUN,
)

print("doScaleVariance:", scaled_config.doScaleVariance)

# List coadd-related datasets actually written by this run.
print("\nCoadd-related datasets in the scaled Step3b run")
print("------------------------------------------------")

for dataset_type in butler.registry.queryDatasetTypes():

    name = dataset_type.name

    if "coadd" not in name.lower():
        continue

    refs = list(
        butler.registry.queryDatasets(
            name,
            collections=SCALED_STEP3B_RUN,
            findFirst=False,
        )
    )

    if refs:
        bands = sorted(
            {
                str(ref.dataId["band"])
                for ref in refs
                if "band" in ref.dataId
            }
        )

        print(
            f"{name:40s} "
            f"N={len(refs):3d}  "
            f"bands={bands}"
        )

doScaleVariance: True

Coadd-related datasets in the scaled Step3b run
------------------------------------------------
assembleCellCoadd_config                 N=  1  bands=[]
assembleCoadd_config                     N=  1  bands=[]
detectCoaddPeaks_config                  N=  1  bands=[]
rewriteCellCoadd_config                  N=  1  bands=[]
selectDeepCoaddVisits_config             N=  1  bands=[]
deep_coadd_visit_selection               N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
selectDeepCoaddVisits_log                N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
selectDeepCoaddVisits_metadata           N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
assembleCoadd_log                        N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
assembleCoadd_metadata                   N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
deep_coadd_input_map                     N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
deep_coadd_predetection                  N=  5  bands=['H', 'J', 'K', 'Y', 'Z']
assembleCellCoadd_log              

### Effect of variance scaling in K

The original and variance-scaled K-band CellCoadds are compared directly. The image plane should remain unchanged because `doScaleVariance` modifies the variance normalisation rather than the measured fluxes.

The median ratio of the new to old variance plane is measured on pixels that are clean in both products. The normalised single-pixel scatter is then recomputed for both versions using the same common pixel sample. If the variance scaling is working as intended, the scaled product should have an NMAD close to unity.

In [14]:
BAND = "K"

data_id = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": BAND,
}

# Original product: doScaleVariance=False
coadd_old = butler.get(
    "deep_coadd_cell",
    dataId=data_id,
    collections=STEP3B_RUN,
)

# New product: doScaleVariance=True
coadd_scaled = butler.get(
    "deep_coadd_cell",
    dataId=data_id,
    collections=SCALED_STEP3B_RUN,
)

exp_old = coadd_old.to_legacy(copy=True)
exp_scaled = coadd_scaled.to_legacy(copy=True)

image_old = np.asarray(exp_old.image.array, dtype=float)
var_old = np.asarray(exp_old.variance.array, dtype=float)
mask_old = np.asarray(exp_old.mask.array)

image_scaled = np.asarray(exp_scaled.image.array, dtype=float)
var_scaled = np.asarray(exp_scaled.variance.array, dtype=float)
mask_scaled = np.asarray(exp_scaled.mask.array)


# Use exactly the same clean pixels for both products.
common_clean = (
    np.isfinite(image_old)
    & np.isfinite(image_scaled)
    & np.isfinite(var_old)
    & np.isfinite(var_scaled)
    & (var_old > 0)
    & (var_scaled > 0)
    & (mask_old == 0)
    & (mask_scaled == 0)
)

print("K-band variance scaling comparison")
print("----------------------------------")
print("Common clean pixels:", common_clean.sum())

# Image should not change.
delta_image = (
    image_scaled[common_clean]
    - image_old[common_clean]
)

print(
    "Median scaled-old image [nJy]:",
    np.median(delta_image),
)

# Direct variance change.
variance_ratio = (
    var_scaled[common_clean]
    / var_old[common_clean]
)

print(
    "Median scaled/old variance    :",
    np.median(variance_ratio),
)


def normalized_scatter(image, variance, selection):
    z = (
        image[selection]
        / np.sqrt(variance[selection])
    )

    med = np.median(z)

    nmad = 1.4826 * np.median(
        np.abs(z - med)
    )

    return med, nmad


med_old, nmad_old = normalized_scatter(
    image_old,
    var_old,
    common_clean,
)

med_scaled, nmad_scaled = normalized_scatter(
    image_scaled,
    var_scaled,
    common_clean,
)

print("\nOriginal K")
print("----------")
print("Median image/sigma :", med_old)
print("NMAD image/sigma   :", nmad_old)

print("\nScaled K")
print("--------")
print("Median image/sigma :", med_scaled)
print("NMAD image/sigma   :", nmad_scaled)

K-band variance scaling comparison
----------------------------------
Common clean pixels: 7665794
Median scaled-old image [nJy]: 0.24334436655044556
Median scaled/old variance    : 1.0

Original K
----------
Median image/sigma : -0.005724684986979705
NMAD image/sigma   : 0.30179346542683433

Scaled K
--------
Median image/sigma : 0.002152393267923206
NMAD image/sigma   : 0.30185075109852505


### Locate the variance-scaled coadd product

Enabling `doScaleVariance` does not change the variance plane of the final `deep_coadd_cell`; the scaled-to-original variance ratio is exactly unity and the normalised single-pixel scatter remains unchanged.

This indicates that the rewritten CellCoadd is not the dataset carrying the detection-stage variance correction. The `deep_coadd` output from `detectCoaddPeaks` is therefore inspected directly, because this is the post-detection exposure on which the empirical variance scaling is applied.

In [15]:
BAND = "K"

data_id = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": BAND,
}

deep_old = butler.get(
    "deep_coadd",
    dataId=data_id,
    collections=STEP3B_RUN,
)

deep_scaled = butler.get(
    "deep_coadd",
    dataId=data_id,
    collections=SCALED_STEP3B_RUN,
)

image_old = np.asarray(deep_old.image.array, dtype=float)
var_old = np.asarray(deep_old.variance.array, dtype=float)
mask_old = np.asarray(deep_old.mask.array)

image_scaled = np.asarray(deep_scaled.image.array, dtype=float)
var_scaled = np.asarray(deep_scaled.variance.array, dtype=float)
mask_scaled = np.asarray(deep_scaled.mask.array)

common_clean = (
    np.isfinite(image_old)
    & np.isfinite(image_scaled)
    & np.isfinite(var_old)
    & np.isfinite(var_scaled)
    & (var_old > 0)
    & (var_scaled > 0)
    & (mask_old == 0)
    & (mask_scaled == 0)
)

variance_ratio = var_scaled[common_clean] / var_old[common_clean]

print("K-band deep_coadd variance comparison")
print("-------------------------------------")
print("Common clean pixels       :", common_clean.sum())
print(
    "Median scaled/old variance:",
    np.median(variance_ratio),
)

for label, image_, var_ in [
    ("Original", image_old, var_old),
    ("Scaled", image_scaled, var_scaled),
]:
    z = image_[common_clean] / np.sqrt(var_[common_clean])
    med = np.median(z)
    nmad = 1.4826 * np.median(np.abs(z - med))

    print(f"\n{label}")
    print("Median image/sigma:", med)
    print("NMAD image/sigma  :", nmad)

K-band deep_coadd variance comparison
-------------------------------------
Common clean pixels       : 7665794
Median scaled/old variance: 0.0995362028488466

Original
Median image/sigma: -0.005724685845391095
NMAD image/sigma  : 0.30179346542683433

Scaled
Median image/sigma: 0.0068223032270024195
NMAD image/sigma  : 0.9567571703377183


### Verify that the variance change is caused by `doScaleVariance`

The original and scaled Step3b runs use the same input data and pipeline configuration except for `detectCoaddPeaks:doScaleVariance`.

To verify that the observed variance correction is introduced specifically by this option, the K-band products are compared both before and after `detectCoaddPeaks`. The upstream `deep_coadd_predetection` image and variance planes should be unchanged between the two runs. In contrast, the post-detection `deep_coadd` variance plane should differ by the empirical scale factor when `doScaleVariance=True`.

In [16]:
BAND = "K"

data_id = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": BAND,
}


def compare_exposures(dataset_type):
    old = butler.get(
        dataset_type,
        dataId=data_id,
        collections=STEP3B_RUN,
    )

    new = butler.get(
        dataset_type,
        dataId=data_id,
        collections=SCALED_STEP3B_RUN,
    )

    img_old = np.asarray(old.image.array, dtype=float)
    img_new = np.asarray(new.image.array, dtype=float)

    var_old = np.asarray(old.variance.array, dtype=float)
    var_new = np.asarray(new.variance.array, dtype=float)

    good = (
        np.isfinite(img_old)
        & np.isfinite(img_new)
        & np.isfinite(var_old)
        & np.isfinite(var_new)
        & (var_old > 0)
        & (var_new > 0)
    )

    img_diff = img_new[good] - img_old[good]
    var_ratio = var_new[good] / var_old[good]

    print(f"\n{dataset_type}")
    print("-" * len(dataset_type))
    print("Pixels compared              :", good.sum())
    print("Median new-old image [nJy]  :", np.median(img_diff))
    print("NMAD new-old image [nJy]    :",
          1.4826 * np.median(np.abs(img_diff - np.median(img_diff))))
    print("Median new/old variance     :", np.median(var_ratio))
    print("16-84% new/old variance    :", np.percentile(var_ratio, [16, 84]))


print("K-band old/new Step3b comparison")
print("================================")

compare_exposures("deep_coadd_predetection")
compare_exposures("deep_coadd")

K-band old/new Step3b comparison

deep_coadd_predetection
-----------------------
Pixels compared              : 10863572
Median new-old image [nJy]  : 0.0
NMAD new-old image [nJy]    : 0.0
Median new/old variance     : 1.0
16-84% new/old variance    : [1. 1.]

deep_coadd
----------
Pixels compared              : 10881460
Median new-old image [nJy]  : 0.26068007946014404
NMAD new-old image [nJy]    : 0.23397100975513457
Median new/old variance     : 0.09953620284910224
16-84% new/old variance    : [0.0995362  0.09953621]


### Confirm the source of the variance correction

The original and scaled runs are identical before `detectCoaddPeaks`: both the image and variance planes of `deep_coadd_predetection` agree exactly.

After `detectCoaddPeaks`, the K-band `deep_coadd` variance in the scaled run is uniformly multiplied by 0.09954 relative to the original run. This agrees closely with the variance factor inferred independently from the original single-pixel noise test. The upstream products are unchanged, so the correction can be attributed directly to enabling `doScaleVariance`.

For K, the normalised single-pixel scatter changes from 0.302 to 0.957 after the correction, showing that the post-detection variance plane is brought close to the measured pixel noise.

### Does Step3c measurement use this corrected deep_coadd variance, or does it use the unscaled CellCoadd variance?

In [17]:
print("Configuration datasets in Step3c")
print("--------------------------------")

step3c_configs = []

for dataset_type in butler.registry.queryDatasetTypes():

    name = dataset_type.name

    if "config" not in name.lower():
        continue

    refs = list(
        butler.registry.queryDatasets(
            name,
            collections=STEP3C_RUN,
            findFirst=False,
        )
    )

    if refs:
        step3c_configs.append(name)


for name in sorted(step3c_configs):
    print(name)

Configuration datasets in Step3c
--------------------------------
deblend_config
deconvolve_config
forcedPhotCoadd_config
mergeDetections_config
mergeMeasurements_config
vircamMeasure_config


### Inspect the VIRCAM measurement configuration

The Step3c VIRCAM measurement task is inspected to determine which coadd representation is used for source measurement.

This is important because `doScaleVariance=True` modifies the post-detection `deep_coadd` variance, whereas the rewritten `deep_coadd_cell` retains the original variance plane. The measurement configuration must therefore be checked before deciding whether the corrected variance will propagate into catalogue flux uncertainties.

In [18]:
vircam_config = butler.get(
    "vircamMeasure_config",
    collections=STEP3C_RUN,
)

print("Config type:", type(vircam_config))

if hasattr(vircam_config, "useCellCoadds"):
    print("useCellCoadds:", vircam_config.useCellCoadds)

print("\nRelevant resolved config entries")
print("--------------------------------")

config_dict = vircam_config.toDict()

def show_relevant(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key

            low = path.lower()

            if any(
                word in low
                for word in [
                    "cell",
                    "coadd",
                    "exposure",
                    "background",
                    "variance",
                ]
            ):
                if not isinstance(value, dict):
                    print(f"{path}: {value}")

            show_relevant(
                value,
                prefix=path,
            )

show_relevant(config_dict)

Config type: <class 'lsst.pipe.tasks.multiBand.MeasureMergedCoaddSourcesConfig'>
useCellCoadds: True

Relevant resolved config entries
--------------------------------
useCellCoadds: True
measurement.plugins.values.base_GaussianFlux.background: 0.0
measurement.plugins.values.base_SdssShape.background: 0.0
measurement.plugins.values.base_LocalBackground.doMeasure: True
measurement.plugins.values.base_LocalBackground.annulusInner: 7.0
measurement.plugins.values.base_LocalBackground.annulusOuter: 15.0
measurement.plugins.values.base_LocalBackground.badMaskPlanes: ['BAD', 'SAT', 'NO_DATA']
measurement.plugins.values.base_LocalBackground.bgIter: 3
measurement.plugins.values.base_LocalBackground.bgRej: 3.0
measurement.plugins.values.base_Variance.doMeasure: True
measurement.plugins.values.base_Variance.scale: 5.0
measurement.plugins.values.base_Variance.mask: ['DETECTED', 'DETECTED_NEGATIVE', 'BAD', 'SAT']
measurement.plugins.values.ip_diffim_DipoleFit.fitBackground: 1
measurement.plugins.va

<string>:3708: FutureWarning: Config field lsst.pipe.base.config.MeasureMergedCoaddSourcesConfigConnections.connections.refCat is deprecated: Reference matching in measureCoaddSources will be removed after v29.


### Determine which variance plane is used for measurement

The VIRCAM Step3c measurement task receives both the post-detection `deep_coadd` exposure and the predetection CellCoadd, while `useCellCoadds=True`.

The post-detection exposure contains the variance correction introduced by `doScaleVariance`, but the CellCoadd variance remains unchanged. The task implementation is therefore inspected to determine which image and variance planes are actually supplied to the photometric measurement plugins.

In [19]:
import inspect

from lsst.pipe.tasks.multiBand import MeasureMergedCoaddSourcesTask

source = inspect.getsource(
    MeasureMergedCoaddSourcesTask.run
)

print(source)

    def run(self, exposure, sources, skyInfo, exposureId, ccdInputs=None,
            sourceTableHandleDict=None, finalizedSourceTableHandleDict=None, finalVisitSummaryHandleDict=None,
            apCorrMap=None):
        """Run measurement algorithms on the input exposure, and optionally populate the
        resulting catalog with extra information.

        Parameters
        ----------
        exposure : `lsst.afw.exposure.Exposure`
            The input exposure on which measurements are to be performed.
        sources :  `lsst.afw.table.SourceCatalog`
            A catalog built from the results of merged detections, or
            deblender outputs.
        parentCatalog : `lsst.afw.table.SourceCatalog`
            Catalog of parent sources corresponding to sources.
        skyInfo : `lsst.pipe.base.Struct`
            A struct containing information about the position of the input exposure within
            a `SkyMap`, the `SkyMap`, its `Wcs`, and its bounding box.
        exp

### Trace the Step3c measurement exposure

`MeasureMergedCoaddSourcesTask.run()` performs all photometric measurements on a single `exposure` object. Therefore, the relevant variance plane is the one attached to that exposure when `run()` is called.

Because `useCellCoadds=True`, the task-level execution code is inspected next to determine whether this exposure is the post-detection `deep_coadd` or is reconstructed from `deep_coadd_cell_predetection` and `deep_coadd_background`.

In [20]:
import inspect

print(
    inspect.getsource(
        MeasureMergedCoaddSourcesTask.runQuantum
    )
)

    def runQuantum(self, butlerQC, inputRefs, outputRefs):
        inputs = butlerQC.get(inputRefs)

        # TODO[DM-47797]: remove this block
        if self.config.doMatchSources:
            refObjLoader = ReferenceObjectLoader([ref.datasetRef.dataId for ref in inputRefs.refCat],
                                                 inputs.pop('refCat'),
                                                 name=self.config.connections.refCat,
                                                 config=self.config.refObjLoader,
                                                 log=self.log)
            self.match.setRefObjLoader(refObjLoader)

        if self.config.useCellCoadds:
            multiple_cell_coadd = inputs.pop("exposure_cells")
            stitched_coadd = multiple_cell_coadd.stitch()
            exposure = stitched_coadd.asExposure()
            background = inputs.pop("background")
            exposure.image -= background.getImage()

            ccdInputs = stitched_coadd.ccds
 

### Variance used by Step3c photometry

With `useCellCoadds=True`, `MeasureMergedCoaddSourcesTask` does not use the post-detection `deep_coadd` for photometric measurement. Instead, it stitches `deep_coadd_cell_predetection`, converts the stitched CellCoadd to an exposure and subtracts `deep_coadd_background`.

All measurement plugins, including flux-uncertainty calculations, therefore operate on the variance plane inherited from `deep_coadd_cell_predetection`.

The predetection CellCoadd is identical between the `doScaleVariance=False` and `doScaleVariance=True` runs. Consequently, enabling detection-stage variance scaling corrects the `deep_coadd` variance used for detection but does not propagate that correction into the CellCoadd-based Step3c photometric uncertainties.

### rewriteCellCoadd

The next useful step is to inspect rewriteCellCoadd, because it sits immediately after detection and is the natural place where a corrected deep_coadd variance might be propagated back into the CellCoadd representation. 

In [21]:
import inspect
import importlib

rewrite_config = butler.get(
    "rewriteCellCoadd_config",
    collections=SCALED_STEP3B_RUN,
)

config_class = type(rewrite_config)
module = importlib.import_module(config_class.__module__)

print("Config class :", config_class)
print("Module       :", config_class.__module__)

print("\nTask classes using this config")
print("------------------------------")

for name, obj in vars(module).items():
    if (
        inspect.isclass(obj)
        and name.endswith("Task")
        and getattr(obj, "ConfigClass", None) is config_class
    ):
        print(name, "->", obj)

Config class : <class 'lsst.pipe.tasks.rewrite_images._rewrite_cell_coadd.RewriteCellCoaddConfig'>
Module       : lsst.pipe.tasks.rewrite_images._rewrite_cell_coadd

Task classes using this config
------------------------------
RewriteCellCoaddTask -> <class 'lsst.pipe.tasks.rewrite_images._rewrite_cell_coadd.RewriteCellCoaddTask'>


### Inspect CellCoadd rewriting after detection

`RewriteCellCoaddTask` runs after source detection and produces the final CellCoadd representation used by later processing.

Because `doScaleVariance=True` modifies the post-detection `deep_coadd` variance but not `deep_coadd_cell_predetection`, the rewrite task is inspected to determine whether it propagates any information from the corrected exposure back into the CellCoadd product.

In [22]:
import inspect

from lsst.pipe.tasks.rewrite_images._rewrite_cell_coadd import (
    RewriteCellCoaddTask,
)

print(
    inspect.getsource(
        RewriteCellCoaddTask.runQuantum
    )
)

    def runQuantum(
        self,
        butlerQC: QuantumContext,
        inputRefs: InputQuantizedConnection,
        outputRefs: OutputQuantizedConnection,
    ) -> None:
        """Do butler IO and transform to provide in memory
        objects for tasks `~Task.run` method.

        Parameters
        ----------
        butlerQC : `QuantumContext`
            A butler which is specialized to operate in the context of a
            `lsst.daf.butler.Quantum`.
        inputRefs : `InputQuantizedConnection`
            Datastructure whose attribute names are the names that identify
            connections defined in corresponding `PipelineTaskConnections`
            class. The values of these attributes are the
            `lsst.daf.butler.DatasetRef` objects associated with the defined
            input/prerequisite connections.
        outputRefs : `OutputQuantizedConnection`
            Datastructure whose attribute names are the names that identify
            connections defined

### Inspect the CellCoadd rewrite logic

`RewriteCellCoaddTask.runQuantum()` only loads the configured inputs and forwards them directly to `run()`. The task's `run()` implementation is therefore inspected to determine whether the post-detection `deep_coadd` image or variance plane is copied back into the final CellCoadd.

In [23]:
print(
    inspect.getsource(
        RewriteCellCoaddTask.run
    )
)

    def run(
        self,
        legacy_cell_coadd: LegacyMultipleCellCoadd,
        sky_map: BaseSkyMap,
        *,
        object_background: BackgroundList | None = None,
        object_mask: LegacyMask | None = None,
        alternate_background_coadd: LegacyExposure | None = None,
    ) -> Struct:
        tract_info = sky_map[legacy_cell_coadd.identifiers.tract]
        patch_bbox = tract_info[legacy_cell_coadd.identifiers.patch].getOuterBBox()
        future_cell_coadd = CellCoadd.from_legacy_cell_coadd(
            legacy_cell_coadd,
            tract_info=tract_info,
            bbox=Box.from_legacy(patch_bbox),
        )
        if getattr(future_cell_coadd, "_opaque_metadata", None) is None:
            future_cell_coadd._opaque_metadata = FitsOpaqueMetadata()
        primary_header = future_cell_coadd._opaque_metadata.headers.setdefault(
            ExtensionKey(), astropy.io.fits.Header()
        )
        primary_header.set("INSTRUME", "LSSTCam")
        primary_header.s

### Why the CellCoadd variance remains unscaled

`RewriteCellCoaddTask` constructs the final CellCoadd directly from the predetection CellCoadd. It then updates the object mask and subtracts the fitted object background from the image.

The task does not modify the variance plane. Consequently, the variance correction applied by `detectCoaddPeaks` to `deep_coadd` is not transferred back into the final `deep_coadd_cell`.

This explains why enabling `doScaleVariance` corrects the variance used for detection while leaving the CellCoadd variance used by Step3c photometric measurement unchanged.

In [24]:
meta = butler.get(
    "detectCoaddPeaks_metadata",
    dataId={
        "skymap": SKYMAP,
        "tract": TRACT,
        "patch": PATCH,
        "band": "K",
    },
    collections=SCALED_STEP3B_RUN,
)

print("Metadata type:", type(meta))

print("\nMetadata:")
print(meta)

Metadata type: <class 'lsst.pipe.base._task_metadata.TaskMetadata'>

Metadata:
scalars={} arrays={} metadata={'detectCoaddPeaks': TaskMetadata(scalars={}, arrays={}, metadata={}), 'detectCoaddPeaks:detection': TaskMetadata(scalars={'__version__': 1.0, 'nodeName': 'uos-1tr5lg4', 'doSmooth': 1.0, 'sigma': 1.9581205228901117, 'smoothingKernelWidth': 15.0, 'nGrow': 5.0}, arrays={'runStartUtc': ['2026-11-03T15:38:56.665600+00:00'], 'runStartCpuTime': [6.282040837], 'runStartUserTime': [5.923696], 'runStartSystemTime': [0.35833699999999996], 'runStartMaxResidentSetSize': [994500608.0], 'runStartMinorPageFaults': [158967.0], 'runStartMajorPageFaults': [0.0], 'runStartBlockInputs': [0.0], 'runStartBlockOutputs': [8.0], 'runStartVoluntaryContextSwitches': [37.0], 'runStartInvoluntaryContextSwitches': [129.0], 'runEndUtc': ['2026-11-03T15:39:31.638625+00:00'], 'runEndCpuTime': [41.165111214], 'runEndUserTime': [40.486632], 'runEndSystemTime': [0.678472], 'runEndMaxResidentSetSize': [1288773632.0

### Optical CellCoadd variance comparison

The same single-pixel variance diagnostic is applied to the reconstructed LSSTCam i-band CellCoadd used by the joint Step3c processing.

The measurement exposure is constructed in the same way as the `useCellCoadds=True` branch of `MeasureMergedCoaddSourcesTask`: the predetection CellCoadd is stitched, converted to an exposure and the detection-stage background model is subtracted. The normalised pixel scatter is then measured on clean pixels.

If the optical CellCoadd gives an NMAD close to unity while the VIRCAM CellCoadds remain substantially below unity, this would indicate that the missing final variance normalisation is particularly important for the VIRCAM processing rather than being a general failure of the CellCoadd measurement path.

In [25]:
OPTICAL_PREDETECTION = "LSSTCam/elais/optical_predetections"
OPTICAL_DETECTION = "LSSTCam/elais/optical_detections"

data_id_i = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": "i",
}

# Same CellCoadd input used by the CellCoadd measurement branch.
cell_i = butler.get(
    "deep_coadd_cell_predetection",
    dataId=data_id_i,
    collections=OPTICAL_PREDETECTION,
)

background_i = butler.get(
    "deep_coadd_background",
    dataId=data_id_i,
    collections=OPTICAL_DETECTION,
)

print("CellCoadd type:", type(cell_i))
print("Background type:", type(background_i))

# Reproduce the useCellCoadds=True measurement exposure.
stitched_i = cell_i.stitch()
exp_i = stitched_i.asExposure()

exp_i.image.array[:] -= background_i.getImage().array

image_i = np.asarray(exp_i.image.array, dtype=float)
variance_i = np.asarray(exp_i.variance.array, dtype=float)
mask_i = np.asarray(exp_i.mask.array)

clean_i = (
    np.isfinite(image_i)
    & np.isfinite(variance_i)
    & (variance_i > 0)
    & (mask_i == 0)
)

sigma_i = np.sqrt(variance_i[clean_i])
z_i = image_i[clean_i] / sigma_i

med_i = np.median(z_i)

nmad_i = 1.4826 * np.median(
    np.abs(z_i - med_i)
)

core_i = np.abs(z_i - med_i) < 5.0 * nmad_i

print("\nLSSTCam i-band CellCoadd variance test")
print("-------------------------------------")
print("Clean pixels            :", clean_i.sum())
print("Clean fraction          :", clean_i.mean())
print("Median image [nJy]      :", np.median(image_i[clean_i]))
print("Median sigma [nJy]      :", np.median(sigma_i))
print("Median image/sigma      :", med_i)
print("NMAD image/sigma        :", nmad_i)
print("Core STD image/sigma    :", np.std(z_i[core_i]))
print("Implied variance scale  :", nmad_i**2)

CellCoadd type: <class 'lsst.cell_coadds._multiple_cell_coadd.MultipleCellCoadd'>
Background type: <class 'lsst.afw.math._backgroundList.BackgroundList'>

LSSTCam i-band CellCoadd variance test
-------------------------------------
Clean pixels            : 9067141
Clean fraction          : 0.8326116620752985
Median image [nJy]      : 0.674412727355957
Median sigma [nJy]      : 2.483305598963901
Median image/sigma      : 0.27117110527794364
NMAD image/sigma        : 1.1468097306448177
Core STD image/sigma    : 1.2379569402096844
Implied variance scale  : 1.3151725583016394


### Check optical detection-stage variance scaling

The predetection i-band CellCoadd gives a normalised pixel scatter moderately above unity. However, the predetection mask does not contain the final object-detection mask, so a `mask == 0` selection can include real sources and overestimate the blank-sky scatter.

The optical detection-stage configuration and post-detection `deep_coadd` are therefore inspected. The final detection mask is used to define blank sky, while the variance ratio between the post-detection and predetection exposures determines whether an empirical variance correction was applied to the optical data.

In [26]:
# Optical detection configuration
opt_config = butler.get(
    "detectCoaddPeaks_config",
    collections=OPTICAL_DETECTION,
)

print("Optical doScaleVariance:", opt_config.doScaleVariance)


# Post-detection optical exposure.
deep_i = butler.get(
    "deep_coadd",
    dataId=data_id_i,
    collections=OPTICAL_DETECTION,
)

deep_image_i = np.asarray(deep_i.image.array, dtype=float)
deep_var_i = np.asarray(deep_i.variance.array, dtype=float)
deep_mask_i = np.asarray(deep_i.mask.array)


# Use the final detection mask to select blank sky,
# but evaluate both predetection and post-detection variance
# on exactly the same pixels.
blank_i = (
    np.isfinite(image_i)
    & np.isfinite(variance_i)
    & np.isfinite(deep_image_i)
    & np.isfinite(deep_var_i)
    & (variance_i > 0)
    & (deep_var_i > 0)
    & (deep_mask_i == 0)
)

var_ratio_i = (
    deep_var_i[blank_i]
    / variance_i[blank_i]
)

print("\nLSSTCam i-band detection-stage comparison")
print("-----------------------------------------")
print("Blank-sky pixels          :", blank_i.sum())
print(
    "Median deep/predet variance:",
    np.median(var_ratio_i),
)


def pixel_nmad(image, variance, selection):
    z = image[selection] / np.sqrt(variance[selection])
    med = np.median(z)
    nmad = 1.4826 * np.median(
        np.abs(z - med)
    )
    return med, nmad


med_pre, nmad_pre = pixel_nmad(
    image_i,
    variance_i,
    blank_i,
)

med_post, nmad_post = pixel_nmad(
    deep_image_i,
    deep_var_i,
    blank_i,
)

print("\nPredetection CellCoadd")
print("  median image/sigma :", med_pre)
print("  NMAD image/sigma   :", nmad_pre)

print("\nPost-detection deep_coadd")
print("  median image/sigma :", med_post)
print("  NMAD image/sigma   :", nmad_post)

Optical doScaleVariance: False

LSSTCam i-band detection-stage comparison
-----------------------------------------
Blank-sky pixels          : 5219316
Median deep/predet variance: 1.0

Predetection CellCoadd
  median image/sigma : 0.031339355145628106
  NMAD image/sigma   : 1.0062557199317983

Post-detection deep_coadd
  median image/sigma : 0.031339355145628106
  NMAD image/sigma   : 1.0062556475390125


### Comparison with LSSTCam

The same blank-sky test was applied to the LSSTCam i-band CellCoadd using the final detection mask. The normalised single-pixel scatter is 1.006, despite `doScaleVariance=False`, and the predetection and post-detection variance planes are identical.

The optical CellCoadd variance is therefore already correctly normalised without an additional detection-stage correction. This contrasts strongly with the VIRCAM bands, where the predetection variance substantially overestimates the measured pixel noise.

The issue is therefore specific to the VIRCAM variance normalisation rather than a general failure of the CellCoadd measurement workflow. For VIRCAM, `doScaleVariance=True` correctly rescales the post-detection `deep_coadd`, but this factor is not propagated to the CellCoadd subsequently used for Step3c photometry.

### Finding the cleanest way to propagate exactly the same scale factor into the CellCoadd

In [27]:
import inspect

from lsst.pipe.tasks.multiBand import DetectCoaddSourcesTask

print(
    inspect.getsource(
        DetectCoaddSourcesTask.run
    )
)

    def run(self, exposure, idFactory, expId, patchInfo=None):
        """Run detection on an exposure.

        First scale the variance plane to match the observed variance
        using ``ScaleVarianceTask``. Then invoke the ``SourceDetectionTask_`` "detection" subtask to
        detect sources.

        Parameters
        ----------
        exposure : `lsst.afw.image.Exposure`
            Exposure on which to detect (may be background-subtracted and scaled,
            depending on configuration).
        idFactory : `lsst.afw.table.IdFactory`
            IdFactory to set source identifiers.
        expId : `int`
            Exposure identifier (integer) for RNG seed.
        patchInfo : `lsst.skymap.PatchInfo`, optional
            Description of the patch geometry.  Only needed if
            `~DetectCoaddSourceConfig.forceExactBinning` is `True`.

        Returns
        -------
        result : `lsst.pipe.base.Struct`
            Results as a struct with attributes:

          

### Recover the variance scale factor

When `doScaleVariance=True`, `DetectCoaddSourcesTask` applies `ScaleVarianceTask` directly to the exposure variance plane. The returned multiplicative factor is stored in the output exposure metadata under `VARIANCE_SCALE`.

This provides the exact factor required to normalise the corresponding CellCoadd variance. Reusing this value is preferable to estimating the variance correction independently a second time.

In [28]:
for run, label in [
    (STEP3B_RUN, "Original"),
    (SCALED_STEP3B_RUN, "Scaled"),
]:
    deep = butler.get(
        "deep_coadd",
        dataId={
            "skymap": SKYMAP,
            "tract": TRACT,
            "patch": PATCH,
            "band": "K",
        },
        collections=run,
    )

    metadata = deep.getMetadata()

    print(f"\n{label} K deep_coadd")
    print("-" * 30)

    if metadata.exists("VARIANCE_SCALE"):
        print(
            "VARIANCE_SCALE:",
            metadata.get("VARIANCE_SCALE"),
        )
    else:
        print("VARIANCE_SCALE: not present")


Original K deep_coadd
------------------------------
VARIANCE_SCALE: not present

Scaled K deep_coadd
------------------------------
VARIANCE_SCALE: 0.0995362015307927


### Variance scale stored by detection

When `doScaleVariance=True`, the post-detection `deep_coadd` stores the empirical correction in its metadata as `VARIANCE_SCALE`.

For the K-band test patch, the stored value is 0.09954, matching the directly measured ratio between the scaled and original `deep_coadd` variance planes. The same factor can therefore be propagated to the CellCoadd variance without rerunning the noise estimator. Now the next question is implementation: does RewriteCellCoaddTask currently receive the post-detection deep_coadd at all?

In [29]:
import inspect
import lsst.pipe.base as pipeBase
import lsst.pipe.tasks.rewrite_images._rewrite_cell_coadd as rewrite_module

print("Connection classes in rewrite_cell_coadd")
print("-----------------------------------------")

for name, obj in vars(rewrite_module).items():
    if (
        inspect.isclass(obj)
        and issubclass(obj, pipeBase.PipelineTaskConnections)
        and obj is not pipeBase.PipelineTaskConnections
    ):
        print(name, "->", obj)

Connection classes in rewrite_cell_coadd
-----------------------------------------
RewriteCellCoaddConnections -> <class 'lsst.pipe.tasks.rewrite_images._rewrite_cell_coadd.RewriteCellCoaddConnections'>


### Inspect `RewriteCellCoaddTask` inputs

The `RewriteCellCoaddConnections` definition is inspected to determine which datasets are available when the final CellCoadd is rewritten.

In particular, this establishes whether the post-detection `deep_coadd`, which contains the stored `VARIANCE_SCALE`, is already an input to the task or would need to be added explicitly.

In [30]:
print(
    inspect.getsource(
        rewrite_module.RewriteCellCoaddConnections
    )
)

class RewriteCellCoaddConnections(
    PipelineTaskConnections,
    dimensions={"patch", "band"},
    defaultTemplates={"legacy_prefix": "legacy_", "future_prefix": ""},
):
    legacy_cell_coadd = cT.Input(
        "deep_coadd_cell_predetection",
        storageClass="MultipleCellCoadd",
        dimensions={"patch", "band"},
        doc="The pre-detection cell coadd to convert.",
    )
    object_background = cT.Input(
        "deep_coadd_background",
        storageClass="Background",
        dimensions={"patch", "band"},
        doc="The background model used to generate the object catalog, to be subtracted from the coadd.",
        minimum=0,
    )
    object_mask = cT.Input(
        "{legacy_prefix}deep_coadd.mask",
        storageClass="Mask",
        dimensions={"patch", "band"},
        doc="The mask to use for the coadd, including the final DETECTED mask plane.",
        minimum=0,
    )
    alternate_background_coadd = cT.Input(
        "pretty_coadd_extra_bg_subtracted",
    

### Resolved `RewriteCellCoadd` connections

The task definition confirms that `RewriteCellCoaddTask` does not currently receive the post-detection `deep_coadd`, so it cannot access the `VARIANCE_SCALE` metadata produced by `detectCoaddPeaks`.

Before modifying the task, the resolved connection configuration is inspected to record the dataset names and prefixes used by the current DP2 pipeline.

In [31]:
rewrite_config = butler.get(
    "rewriteCellCoadd_config",
    collections=SCALED_STEP3B_RUN,
)

config_dict = rewrite_config.toDict()

print("RewriteCellCoadd resolved connections")
print("-------------------------------------")

for key, value in config_dict.get("connections", {}).items():
    print(f"{key:30s}: {value}")

print("\ndo_alternate_background_fit:",
      rewrite_config.do_alternate_background_fit)

RewriteCellCoadd resolved connections
-------------------------------------
legacy_cell_coadd             : deep_coadd_cell_predetection
object_background             : deep_coadd_background
object_mask                   : deep_coadd.mask
alternate_background_coadd    : pretty_coadd_extra_bg_subtracted
sky_map                       : skyMap
future_cell_coadd             : deep_coadd_cell
legacy_prefix                 : legacy_
future_prefix                 : 

do_alternate_background_fit: False


In [32]:
data_id_k = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": "K",
}

deep_metadata = butler.get(
    "deep_coadd.metadata",
    dataId=data_id_k,
    collections=SCALED_STEP3B_RUN,
)

print("Type:", type(deep_metadata))
print("VARIANCE_SCALE:", deep_metadata.get("VARIANCE_SCALE"))

Type: <class 'lsst.daf.base.PropertyList'>
VARIANCE_SCALE: 0.0995362015307927


In [33]:
# Use the already loaded K-band CellCoadd.
future_test = coadd_old

print("CellCoadd type:", type(future_test))

print("\nVariance-related attributes:")
for name in dir(future_test):
    if "var" in name.lower():
        print(name)

if hasattr(future_test, "variance"):
    print("\nvariance type :", type(future_test.variance))
    if hasattr(future_test.variance, "array"):
        print("variance shape:", future_test.variance.array.shape)

CellCoadd type: <class 'lsst.images.cells._coadd.CellCoadd'>

Variance-related attributes:
_variance
variance

variance type : <class 'lsst.images._image.Image'>
variance shape: (3300, 3300)


### Proposed CellCoadd variance propagation

The CellCoadd variance plane is directly writable through `CellCoadd.variance.array`. The smallest modification is therefore to provide `RewriteCellCoaddTask` with the metadata component of the post-detection `deep_coadd`.

When `VARIANCE_SCALE` is present, the same factor already determined by `ScaleVarianceTask` is applied to the CellCoadd variance during rewriting. If the metadata key is absent, the variance is left unchanged. This preserves the existing behaviour for processing in which `doScaleVariance=False`.

Only the variance plane is modified; the image, mask, background, PSF and aperture-correction information are unaffected.

In [34]:
print(inspect.getfile(rewrite_module))

/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/pipe_tasks/gc9673a5fcf+9900def6f8/python/lsst/pipe/tasks/rewrite_images/_rewrite_cell_coadd.py


In [35]:
import os

print("PIPE_TASKS_DIR =", os.environ.get("PIPE_TASKS_DIR"))

PIPE_TASKS_DIR = /local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/pipe_tasks/gc9673a5fcf+9900def6f8


In [36]:
from pathlib import Path
import os

pipe_tasks_dir = Path(os.environ["PIPE_TASKS_DIR"])

print("PIPE_TASKS_DIR:")
print(pipe_tasks_dir)

print("\nResolved path:")
print(pipe_tasks_dir.resolve())

print("\n.git exists here:", (pipe_tasks_dir / ".git").exists())

print("\nParent directories containing .git:")
for parent in [pipe_tasks_dir, *pipe_tasks_dir.parents]:
    if (parent / ".git").exists():
        print(parent)

PIPE_TASKS_DIR:
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/pipe_tasks/gc9673a5fcf+9900def6f8

Resolved path:
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/pipe_tasks/gc9673a5fcf+9900def6f8

.git exists here: False

Parent directories containing .git:
/local/scratch/es3e23/projects/lsst-ir-fusion


In [37]:
from pathlib import Path
import subprocess

repo_root = Path("/local/scratch/es3e23/projects/lsst-ir-fusion")
source_file = Path(inspect.getfile(rewrite_module))

relative = source_file.relative_to(repo_root)

print("Relative path:")
print(relative)

result = subprocess.run(
    ["git", "-C", str(repo_root), "ls-files", str(relative)],
    capture_output=True,
    text=True,
)

print("\nTracked by lsst-ir-fusion:")
print(result.stdout if result.stdout else "No")

Relative path:
install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/pipe_tasks/gc9673a5fcf+9900def6f8/python/lsst/pipe/tasks/rewrite_images/_rewrite_cell_coadd.py

Tracked by lsst-ir-fusion:
No


In [38]:
import subprocess

result = subprocess.run(
    [
        "find",
        "/local/scratch/es3e23/projects/lsst-ir-fusion",
        "-path", "*/_rewrite_cell_coadd.py",
        "-not", "-path", "*/install/source/*",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout if result.stdout else "No source copy found")

No source copy found


In [39]:
import os
from pathlib import Path

obs_vista = Path(os.environ["OBS_VISTA_DIR"])

print("OBS_VISTA_DIR:")
print(obs_vista)

print("\nPython task files:")
for path in sorted((obs_vista / "python").rglob("*.py")):
    print(path.relative_to(obs_vista))

OBS_VISTA_DIR:
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/obs_vista/24.0.0.1

Python task files:
python/__init__.py
python/lsstuk/__init__.py
python/lsstuk/obs/__init__.py
python/lsstuk/obs/vista/__init__.py
python/lsstuk/obs/vista/_instrument.py
python/lsstuk/obs/vista/makeVircamRawVisitInfo.py
python/lsstuk/obs/vista/rawFormatter.py
python/lsstuk/obs/vista/scaleVarianceCalexp.py
python/lsstuk/obs/vista/translators/__init__.py
python/lsstuk/obs/vista/translators/vircam.py
python/lsstuk/obs/vista/vircamFilters.py
python/lsstuk/obs/vista/vircamIsrTask.py


In [40]:
import subprocess

result = subprocess.run(
    [
        "find",
        "/local/scratch/es3e23/projects/lsst-ir-fusion",
        "-name", "DRP-DP2.yaml",
        "-not", "-path", "*/install/source/*",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout if result.stdout else "No source copy found")

No source copy found


In [41]:
import subprocess

result = subprocess.run(
    [
        "find",
        "/local/scratch/es3e23",
        "-type", "f",
        "-path", "*/pipelines/DRP-DP2.yaml",
        "-not", "-path", "*/install/source/*",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout if result.stdout else "No editable obs_vista checkout found")

No editable obs_vista checkout found


In [42]:
print("Dataset types containing 'warp'")
print("--------------------------------")

for dt in sorted(
    butler.registry.queryDatasetTypes(),
    key=lambda x: x.name
):
    if "warp" in dt.name.lower():
        print(dt.name)

Dataset types containing 'warp'
--------------------------------
compare_warp_artifact_mask
direct_warp
direct_warp_masked_fraction
direct_warp_noise0
makeDirectWarp_config
makeDirectWarp_log
makeDirectWarp_metadata
makePsfMatchedWarp_config
makePsfMatchedWarp_log
makePsfMatchedWarp_metadata
psf_matched_warp


In [43]:
dt = butler.registry.getDatasetType("direct_warp")

print("direct_warp")
print("-----------")
print("Storage class:", dt.storageClass_name)
print("Dimensions   :", sorted(dt.dimensions.names))

print("\nK-band direct_warp collections")
print("------------------------------")

refs = list(
    butler.registry.queryDatasets(
        "direct_warp",
        collections="*",
        where="band='K'",
    )
)

print("Number of K direct_warps:", len(refs))

for ref in refs[:20]:
    print(ref.run, ref.dataId)

direct_warp
-----------
Storage class: ExposureF
Dimensions   : ['band', 'day_obs', 'instrument', 'patch', 'physical_filter', 'skymap', 'tract', 'visit']

K-band direct_warp collections
------------------------------
Number of K direct_warps: 610
videoStep3a/20261018T142051Z {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 215044, band: 'K', day_obs: 20101111, physical_filter: 'VIRCAM-Ks'}
videoStep3a/20261018T142051Z {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 198704, band: 'K', day_obs: 20101016, physical_filter: 'VIRCAM-Ks'}
videoStep3a/20261018T142051Z {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 187463, band: 'K', day_obs: 20100923, physical_filter: 'VIRCAM-Ks'}
videoStep3a/20261018T142051Z {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 216818, band: 'K', day_obs: 20101114, physical_filter: 'VIRCAM-Ks'}
videoStep3a/20261018T142051Z {instrument: 'VIRCAM

In [44]:
# Inspect one representative K-band direct warp.

warp_ref = refs[0]
warp_k = butler.get(warp_ref)

print("Visit:", warp_ref.dataId["visit"])
print("Run  :", warp_ref.run)
print("Type :", type(warp_k))
print("BBox :", warp_k.getBBox())

print("\nMask planes")
print("-----------")
for name, bit in sorted(
    warp_k.mask.getMaskPlaneDict().items(),
    key=lambda x: x[1]
):
    print(f"{name:20s} bit={bit}")

image_w = np.asarray(warp_k.image.array, dtype=float)
var_w = np.asarray(warp_k.variance.array, dtype=float)
mask_w = np.asarray(warp_k.mask.array)

finite = (
    np.isfinite(image_w)
    & np.isfinite(var_w)
    & (var_w > 0)
)

print("\nBasic statistics")
print("----------------")
print("Finite pixels          :", finite.sum())
print("Finite fraction        :", finite.mean())
print("Median image [nJy]     :", np.median(image_w[finite]))
print("Median variance [nJy²] :", np.median(var_w[finite]))
print("Median sigma [nJy]     :", np.median(np.sqrt(var_w[finite])))

print("\nMask occupancy")
print("--------------")
for name in warp_k.mask.getMaskPlaneDict():
    bitmask = warp_k.mask.getPlaneBitMask(name)
    frac = np.mean((mask_w & bitmask) != 0)
    if frac > 0:
        print(f"{name:20s}: {100*frac:8.3f}%")

Visit: 215044
Run  : videoStep3a/20261018T142051Z
Type : <class 'lsst.afw.image._exposure.ExposureF'>
BBox : (minimum=(20594, 20594), maximum=(24405, 24405))

Mask planes
-----------
BAD                  bit=0
SAT                  bit=1
INTRP                bit=2
CR                   bit=3
EDGE                 bit=4
DETECTED             bit=5
DETECTED_NEGATIVE    bit=6
SUSPECT              bit=7
NO_DATA              bit=8
VIGNETTED            bit=9
STREAK               bit=10
CLIPPED              bit=11
REJECTED             bit=12
INEXACT_PSF          bit=13
NOT_DEBLENDED        bit=14
SENSOR_EDGE          bit=15
UNMASKEDNAN          bit=16

Basic statistics
----------------
Finite pixels          : 1470733
Finite fraction        : 0.10121107861736671
Median image [nJy]     : -0.10608295351266861
Median variance [nJy²] : 71959.265625
Median sigma [nJy]     : 268.2522425348948

Mask occupancy
--------------
BAD                 :    1.266%
DETECTED            :    0.099%
EDGE            

In [45]:
clean_w = (
    np.isfinite(image_w)
    & np.isfinite(var_w)
    & (var_w > 0)
    & (mask_w == 0)
)

sigma_w = np.sqrt(var_w[clean_w])
z_w = image_w[clean_w] / sigma_w

med_w = np.median(z_w)
nmad_w = 1.4826 * np.median(np.abs(z_w - med_w))

core_w = np.abs(z_w - med_w) < 5.0 * nmad_w

print("K direct_warp single-pixel variance test")
print("----------------------------------------")
print("Visit                    :", warp_ref.dataId["visit"])
print("Clean pixels             :", clean_w.sum())
print("Clean fraction of image  :", clean_w.mean())
print("Median image [nJy]       :", np.median(image_w[clean_w]))
print("Median sigma [nJy]       :", np.median(sigma_w))
print("Median image/sigma       :", med_w)
print("NMAD image/sigma         :", nmad_w)
print("Core STD image/sigma     :", np.std(z_w[core_w]))
print("Implied variance scale   :", nmad_w**2)

K direct_warp single-pixel variance test
----------------------------------------
Visit                    : 215044
Clean pixels             : 1315888
Clean fraction of image  : 0.09055514754863693
Median image [nJy]       : -0.8352900743484497
Median sigma [nJy]       : 262.13668309679974
Median image/sigma       : -0.0028599239096712345
NMAD image/sigma         : 0.2796581243171304
Core STD image/sigma     : 0.2841900010501878
Implied variance scale   : 0.07820866649657555


In [46]:
data_id_k = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": "K",
}

for dtype in ["assembleCoadd_log", "assembleCellCoadd_log"]:
    print(f"\n{dtype}")
    print("-" * len(dtype))

    try:
        logs = butler.get(
            dtype,
            dataId=data_id_k,
            collections=STEP3B_RUN,
        )
    except Exception as e:
        print("Could not load:", e)
        continue

    found = False

    for record in logs:
        # Butler log records usually expose message directly,
        # but this keeps the cell tolerant of different representations.
        msg = getattr(record, "message", None)
        if msg is None:
            try:
                msg = record.getMessage()
            except Exception:
                msg = str(record)

        low = msg.lower()

        if any(
            word in low
            for word in [
                "variance",
                "renormal",
                "scale",
                "rescal",
            ]
        ):
            print(msg)
            found = True

    if not found:
        print("No variance/scale-related log messages found.")


assembleCoadd_log
-----------------
Renormalizing variance by 0.465011
Renormalizing variance by 0.461109
Renormalizing variance by 0.438128
Renormalizing variance by 0.424777
Renormalizing variance by 0.446548
Renormalizing variance by 0.436624
Renormalizing variance by 0.519946
Renormalizing variance by 0.512850
Renormalizing variance by 0.504064
Renormalizing variance by 0.420511
Renormalizing variance by 0.526027
Renormalizing variance by 0.459161
Renormalizing variance by 0.459980
Renormalizing variance by 0.467598
Renormalizing variance by 0.487708
Renormalizing variance by 0.488318
Renormalizing variance by 0.482458
Renormalizing variance by 0.450038
Renormalizing variance by 0.459330
Renormalizing variance by 0.498235
Renormalizing variance by 0.505337
Renormalizing variance by 0.524718
Renormalizing variance by 0.451329
Renormalizing variance by 0.424872
Renormalizing variance by 0.426422
Renormalizing variance by 0.474859
Renormalizing variance by 0.541428
Renormalizing vari

In [47]:
import subprocess
from pathlib import Path
import os

stack_root = Path(os.environ["PIPE_TASKS_DIR"]).parents[1]

result = subprocess.run(
    [
        "grep",
        "-R",
        "-n",
        "-B", "12",
        "-A", "18",
        "Renormalizing variance by",
        str(stack_root),
    ],
    capture_output=True,
    text=True,
)

print(result.stdout if result.stdout else result.stderr)

/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/meas_algorithms/gab846f1091+17a9a7c0ba/tests/.tests/pytest-meas_algorithms.xml-htmlcov/z_1e4b30e7f401bbe5_scaleVariance_py.html-231-    <p class="pln"><span class="n"><a id="t147" href="#t147">147</a></span><span class="t"><span class="str">        The task calculates and applies the pixel-based correction unless</span>&nbsp;</span><span class="r"></span></p>
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/meas_algorithms/gab846f1091+17a9a7c0ba/tests/.tests/pytest-meas_algorithms.xml-htmlcov/z_1e4b30e7f401bbe5_scaleVariance_py.html-232-    <p class="pln"><span class="n"><a id="t148" href="#t148">148</a></span><span class="t"><span class="str">        it is over the ``config.limit`` threshold. In this case, the image-based</span>&nbsp;</span><span class="r"></span></p>
/local/scratch/es

### Reproduce Rubin variance scaling on one direct warp

The previous blank-sky NMAD estimate is not necessarily identical to the estimator used internally by `ScaleVarianceTask`. To make a direct comparison with the legacy `assembleCoadd` processing, the same Rubin task and the saved `scaleWarpVariance` configuration are applied to the representative K-band direct warp.

`computeScaleFactors` is used so that the diagnostic does not modify the stored or in-memory variance plane.

In [48]:
from lsst.meas.algorithms import ScaleVarianceTask

assemble_config = butler.get(
    "assembleCoadd_config",
    collections=STEP3B_RUN,
)

print("doScaleWarpVariance:",
      assemble_config.doScaleWarpVariance)

print("ScaleVariance limit:",
      assemble_config.scaleWarpVariance.limit)

scale_task = ScaleVarianceTask(
    config=assemble_config.scaleWarpVariance
)

factors = scale_task.computeScaleFactors(
    warp_k.getMaskedImage()
)

print("\nVisit:", warp_ref.dataId["visit"])
print("Rubin pixelFactor :", factors.pixelFactor)
print("Rubin imageFactor :", factors.imageFactor)

print("\nOur NMAD factor   :", nmad_w**2)

doScaleWarpVariance: True
ScaleVariance limit: 10.0

Visit: 215044
Rubin pixelFactor : 0.07650686544066589
Rubin imageFactor : 0.09584476154836112

Our NMAD factor   : 0.07820866649657555


### Direct-warp variance normalization

The Rubin `ScaleVarianceTask` estimator was applied directly to the representative K-band warp without modifying it. The resulting pixel-based factor is 0.0765, in excellent agreement with the independently measured blank-sky factor of 0.0782.

The variance over-normalisation therefore already exists in the VIRCAM direct warps, before coaddition. The appropriate solution should consequently correct the warp variance before it is propagated into the CellCoadd, rather than applying a posterior correction to the final CellCoadd or photometric catalogue.

In [49]:
print("assembleCoadd input configuration")
print("---------------------------------")

print("warpType             :", assemble_config.warpType)
print("doScaleWarpVariance  :", assemble_config.doScaleWarpVariance)
print("calcErrorFromInputVariance:",
      assemble_config.calcErrorFromInputVariance)

print("\nResolved connections")
print("--------------------")

config_dict = assemble_config.toDict()

for key, value in config_dict.get("connections", {}).items():
    print(f"{key:35s}: {value}")

assembleCoadd input configuration
---------------------------------
warpType             : direct
doScaleWarpVariance  : True
calcErrorFromInputVariance: True

Resolved connections
--------------------
inputWarps                         : direct_warp
skyMap                             : skyMap
selectedVisits                     : deep_coadd_visit_selection
brightObjectMask                   : brightObjectMask
coaddExposure                      : deep_coadd_predetection
nImage                             : deep_coadd_n_image
inputMap                           : deep_coadd_input_map
psfMatchedWarps                    : psf_matched_warp
templateCoadd                      : {outputCoaddName}CoaddPsfMatched
artifactMasks                      : compare_warp_artifact_mask
inputCoaddName                     : deep
outputCoaddName                    : deep
warpType                           : direct
warpTypeSuffix                     : 


In [50]:
logs_k = butler.get(
    "assembleCoadd_log",
    dataId=data_id_k,
    collections=STEP3B_RUN,
)

print("Log type:", type(logs_k))
print("Number of records:", len(logs_k))

print("\nFirst 5 records")
print("----------------")

for i, record in enumerate(logs_k[:5]):
    print(f"\nRecord {i}")
    print("message:", getattr(record, "message", None))

    # Show structured context attached to the log record.
    for attr in ["MDC", "mdc", "name", "levelname"]:
        if hasattr(record, attr):
            print(f"{attr}:", getattr(record, attr))

Log type: <class 'lsst.daf.butler.logging.ButlerLogRecords'>
Number of records: 1021

First 5 records
----------------

Record 0
message: Preparing execution of quantum for label=assembleCoadd dataId={band: 'K', skymap: 'lsst_cells_v2', tract: 2877, patch: 77}.
MDC: {'LABEL': "assembleCoadd:{band: 'K', skymap: 'lsst_cells_v2', tract: 2877, patch: 77}", 'RUN': 'videoStep3b/20261018T154842Z'}
name: lsst.pipe.base.single_quantum_executor
levelname: INFO

Record 1
message: Constructing task and executing quantum for label=assembleCoadd dataId={band: 'K', skymap: 'lsst_cells_v2', tract: 2877, patch: 77}.
MDC: {'LABEL': "assembleCoadd:{band: 'K', skymap: 'lsst_cells_v2', tract: 2877, patch: 77}", 'RUN': 'videoStep3b/20261018T154842Z'}
name: lsst.pipe.base.single_quantum_executor
levelname: INFO

Record 2
message: Weight of deepCoadd_directWarp {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 203180, band: 'K', day_obs: 20101024, physical_filter: 'VIRCAM-Ks'} = 0

In [51]:
visit_target = 215044

# Find every log record mentioning this visit.
indices = [
    i for i, record in enumerate(logs_k)
    if str(visit_target) in getattr(record, "message", "")
]

print("Records mentioning visit", visit_target, ":", indices)

for idx in indices:
    print("\n" + "=" * 90)
    print(f"Context around log record {idx}")
    print("=" * 90)

    start = max(0, idx - 8)
    stop = min(len(logs_k), idx + 12)

    for j in range(start, stop):
        record = logs_k[j]
        print(
            f"{j:4d} | "
            f"{record.name:35s} | "
            f"{record.message}"
        )

Records mentioning visit 215044 : [110, 312]

Context around log record 110
 102 | lsst.assembleCoadd                  | Weight of deepCoadd_directWarp {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 187224, band: 'K', day_obs: 20100922, physical_filter: 'VIRCAM-Ks'} = 0.000
 103 | lsst.assembleCoadd                  | Weight of deepCoadd_directWarp {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 187053, band: 'K', day_obs: 20100922, physical_filter: 'VIRCAM-Ks'} = 0.000
 104 | lsst.assembleCoadd                  | Weight of deepCoadd_directWarp {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 143431, band: 'K', day_obs: 20100722, physical_filter: 'VIRCAM-Ks'} = 0.000
 105 | lsst.assembleCoadd                  | Weight of deepCoadd_directWarp {instrument: 'VIRCAM', skymap: 'lsst_cells_v2', tract: 2877, patch: 77, visit: 228873, band: 'K', day_obs: 20101207, physical_filter: 'VIRCAM-Ks'} = 0.000


In [52]:
make_warp_config = butler.get(
    "makeDirectWarp_config",
    collections="videoStep3a/20261018T142051Z",
)

config_dict = make_warp_config.toDict()

print("makeDirectWarp variance / scaling-related configuration")
print("-------------------------------------------------------")

def show_relevant(d, prefix=""):
    for key, value in d.items():
        path = f"{prefix}.{key}" if prefix else key

        if isinstance(value, dict):
            show_relevant(value, path)
        else:
            low = path.lower()
            if any(
                word in low
                for word in [
                    "variance",
                    "scale",
                    "weight",
                    "noise",
                    "background",
                ]
            ):
                print(f"{path:60s}: {value}")

show_relevant(config_dict)

makeDirectWarp variance / scaling-related configuration
-------------------------------------------------------
numberOfNoiseRealizations                                   : 1
useMedianVariance                                           : True
doRevertOldBackground                                       : False
doApplyNewBackground                                        : False
doApplyFlatBackgroundRatio                                  : False
select.maxScaledSizeScatter                                 : 0.022
select.maxPsfApCorrSigmaScaledDelta                         : 0.13
inputRecorder.saveCcdWeights                                : True
connections.background_revert_list                          : calexpBackground
connections.background_apply_list                           : skyCorr
connections.background_to_photometric_ratio_list            : background_to_photometric_ratio


In [53]:
cell_config = butler.get(
    "assembleCellCoadd_config",
    collections=STEP3B_RUN,
)

print("assembleCellCoadd configuration")
print("--------------------------------")

print("calc_error_from_input_variance:",
      cell_config.calc_error_from_input_variance)

config_dict = cell_config.toDict()

print("\nResolved connections")
print("--------------------")
for key, value in config_dict.get("connections", {}).items():
    print(f"{key:35s}: {value}")

assembleCellCoadd configuration
--------------------------------
calc_error_from_input_variance: True

Resolved connections
--------------------
inputWarps                         : direct_warp
maskedFractionWarps                : direct_warp_masked_fraction
artifactMasks                      : compare_warp_artifact_mask
visitSummaryList                   : finalVisitSummary
skyMap                             : skyMap
multipleCellCoadd                  : deep_coadd_cell_predetection
inputMap                           : {inputWarpName}Coadd_inputMap
inputWarpName                      : deep
outputCoaddSuffix                  : Cell


In [54]:
import subprocess
from pathlib import Path
import os

stack_root = Path(os.environ["PIPE_TASKS_DIR"]).parents[1]

result = subprocess.run(
    [
        "grep",
        "-R",
        "-n",
        "--include=*.py",
        "ScaleVarianceTask",
        str(stack_root),
    ],
    capture_output=True,
    text=True,
)

print(result.stdout if result.stdout else "No occurrences found")

/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/meas_algorithms/gab846f1091+17a9a7c0ba/tests/test_variance.py:32:    ScaleVarianceTask,
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/meas_algorithms/gab846f1091+17a9a7c0ba/tests/test_variance.py:119:        """Test that the ScaleVarianceTask scales the variance plane correctly."""
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/meas_algorithms/gab846f1091+17a9a7c0ba/tests/test_variance.py:120:        task = ScaleVarianceTask()
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/meas_algorithms/gab846f1091+17a9a7c0ba/python/lsst/meas/algorithms/scaleVariance.py:29:__all__ = ["ScaleVarianceConfig", "ScaleVarianceTask", "ExceedsMaxVaria

In [55]:
import inspect
import lsst.drp.tasks.assemble_coadd as ac

source = inspect.getsource(ac.AssembleCoaddTask)

for line_no, line in enumerate(source.splitlines(), start=1):
    if (
        "doScaleWarpVariance" in line
        or "scaleWarpVariance" in line
        or "Renormalizing variance" in line
    ):
        start = max(1, line_no - 12)
        stop = line_no + 18

        print("\n" + "=" * 90)
        print(f"Context around AssembleCoaddTask line {line_no}")
        print("=" * 90)

        lines = source.splitlines()
        for j in range(start, min(stop, len(lines)) + 1):
            print(f"{j:5d}: {lines[j-1]}")

In [56]:
import inspect
from pathlib import Path
import lsst.drp.tasks.assemble_coadd as ac

assemble_file = Path(inspect.getfile(ac))

print("File:")
print(assemble_file)

print("\nOccurrences:")
print("------------")

lines = assemble_file.read_text().splitlines()

for i, line in enumerate(lines, start=1):
    if (
        "doScaleWarpVariance" in line
        or "scaleWarpVariance" in line
        or "ScaleVarianceTask" in line
    ):
        print(f"{i:5d}: {line}")

File:
/local/scratch/es3e23/projects/lsst-ir-fusion/install/source/v30_0_10/conda/envs/lsst-scipipe-12.3.0-exact/share/eups/Linux64/drp_tasks/g2b15b11b72+34594061e0/python/lsst/drp/tasks/assemble_coadd.py

Occurrences:
------------
   50: from lsst.meas.algorithms import AccumulatorMeanStack, MaskStreaksTask, ScaleVarianceTask, SourceDetectionTask
 1416:     doScaleWarpVariance = pexConfig.Field(
 1421:     scaleWarpVariance = pexConfig.ConfigurableField(
 1422:         target=ScaleVarianceTask,
 1574:         if self.config.doScaleWarpVariance:
 1575:             self.makeSubtask("scaleWarpVariance")
 2045:         if self.config.doScaleWarpVariance:
 2047:                 self.scaleWarpVariance.run(mi)


In [57]:
start = 2028
stop = 2065

for i in range(start, stop + 1):
    print(f"{i:5d}: {lines[i-1]}")

 2028: 
 2029:         Returns
 2030:         -------
 2031:         warp : `lsst.afw.image.Exposure`
 2032:             Exposure of the image difference between the warp and template.
 2033:         """
 2034:         # If the PSF-Matched warp did not exist for this direct warp
 2035:         # None is holding its place to maintain order in Gen 3
 2036:         if warpRef is None:
 2037:             return None
 2038: 
 2039:         warp = warpRef.get(parameters={"bbox": templateCoadd.getBBox()})
 2040:         # direct image scaler OK for PSF-matched Warp.
 2041:         # Remove in DM-49083.
 2042:         if imageScaler is not None:
 2043:             imageScaler.scaleMaskedImage(warp.getMaskedImage())
 2044:         mi = warp.getMaskedImage()
 2045:         if self.config.doScaleWarpVariance:
 2046:             try:
 2047:                 self.scaleWarpVariance.run(mi)
 2048:             except Exception as exc:
 2049:                 self.log.warning("Unable to rescale variance 

IndexError: list index out of range

In [ ]:
scale_indices = [
    i for i, record in enumerate(logs_k)
    if "Renormalizing variance by" in getattr(record, "message", "")
]

print("Number of variance-scaling messages:", len(scale_indices))

for idx in scale_indices[:10]:
    record = logs_k[idx]

    print("\n" + "=" * 80)
    print("Index :", idx)
    print("Logger:", record.name)
    print("Scale :", record.message)

    # A little context immediately before the scale message.
    for j in range(max(0, idx - 3), idx):
        print(
            f"  {j:4d} | "
            f"{logs_k[j].name:45s} | "
            f"{logs_k[j].message}"
        )

In [ ]:
print("makeDirectWarp resolved connections")
print("-----------------------------------")

for key, value in make_warp_config.toDict().get("connections", {}).items():
    print(f"{key:40s}: {value}")

In [ ]:
visit_target = 215044

dt_calexp = butler.registry.getDatasetType("calexp")

print("calexp")
print("------")
print("Storage class:", dt_calexp.storageClass_name)
print("Dimensions   :", sorted(dt_calexp.dimensions.names))

calexp_refs = list(
    butler.registry.queryDatasets(
        "calexp",
        collections="*",
        where=(
            f"instrument='VIRCAM' "
            f"AND visit={visit_target} "
            f"AND band='K'"
        ),
    )
)

print("\nNumber of calexps for visit", visit_target, ":", len(calexp_refs))

for ref in calexp_refs:
    print(ref.run, ref.dataId)

### Calexp variance normalisation

The variance normalisation is measured in the detector-level K-band `calexp`s contributing to visit 215044, before resampling into the `direct_warp`.

The same Rubin `ScaleVarianceTask` estimator used for the warp diagnostic is applied without modifying the exposures. If the `calexp` factors are already substantially below unity, the variance overestimation originates before `MakeDirectWarpTask`. If they are near unity while the direct warp requires a factor of about 0.077, the problem is introduced during warping.

In [ ]:
from lsst.meas.algorithms import ScaleVarianceTask

scale_task_calexp = ScaleVarianceTask()

print("K calexp variance tests — visit", visit_target)
print("=" * 60)

for ref in calexp_refs:
    exp = butler.get(ref)

    factors = scale_task_calexp.computeScaleFactors(
        exp.getMaskedImage()
    )

    image_c = np.asarray(exp.image.array, dtype=float)
    var_c = np.asarray(exp.variance.array, dtype=float)
    mask_c = np.asarray(exp.mask.array)

    clean_c = (
        np.isfinite(image_c)
        & np.isfinite(var_c)
        & (var_c > 0)
        & (mask_c == 0)
    )

    z_c = image_c[clean_c] / np.sqrt(var_c[clean_c])
    med_c = np.median(z_c)
    nmad_c = 1.4826 * np.median(
        np.abs(z_c - med_c)
    )

    print(f"\nDetector {ref.dataId['detector']}")
    print("-" * 30)
    print("Clean pixels        :", clean_c.sum())
    print("Median image/sigma  :", med_c)
    print("NMAD image/sigma    :", nmad_c)
    print("NMAD variance factor:", nmad_c**2)
    print("Rubin pixelFactor   :", factors.pixelFactor)
    print("Rubin imageFactor   :", factors.imageFactor)

In [ ]:
STEP1_RUN = "videoStep1/20261015T160027Z"

print("Configuration datasets in Step1")
print("--------------------------------")

for dt in sorted(
    butler.registry.queryDatasetTypes(),
    key=lambda x: x.name
):
    if not dt.name.endswith("_config"):
        continue

    refs_cfg = list(
        butler.registry.queryDatasets(
            dt.name,
            collections=STEP1_RUN,
        )
    )

    if refs_cfg:
        print(f"{dt.name:40s}  N={len(refs_cfg)}")

In [ ]:
from lsst.meas.algorithms import ScaleVarianceTask

# Use the same ScaleVariance configuration as assembleCoadd.
scale_calexp_task = ScaleVarianceTask(
    config=assemble_config.scaleWarpVariance
)

print("In-memory calexp variance-normalisation test")
print("=" * 60)

for ref in calexp_refs:

    # Fresh exposure from Butler, then clone so the stored dataset is untouched.
    exp_original = butler.get(ref)
    exp_scaled = exp_original.clone()

    # Factor before correction.
    before = scale_calexp_task.computeScaleFactors(
        exp_scaled.getMaskedImage()
    )

    # Apply Rubin's standard correction in place.
    applied_factor = scale_calexp_task.run(
        exp_scaled.getMaskedImage()
    )

    # Measure again after correction.
    after = scale_calexp_task.computeScaleFactors(
        exp_scaled.getMaskedImage()
    )

    print(f"\nDetector {ref.dataId['detector']}")
    print("-" * 30)
    print("Before pixelFactor :", before.pixelFactor)
    print("Applied factor     :", applied_factor)
    print("After pixelFactor  :", after.pixelFactor)
    print("After imageFactor  :", after.imageFactor)

In [ ]:
from lsst.daf.butler import Butler

TEST_CAL_RUN = "varianceTest/calexpScaledK215044"

butler_write = Butler(
    REPO,
    writeable=True,
)

butler_write.registry.registerRun(TEST_CAL_RUN)

print("Writing corrected calexps to:")
print(TEST_CAL_RUN)
print("-" * 60)

for ref in calexp_refs:
    exp_scaled = butler.get(ref).clone()

    factor = scale_calexp_task.run(
        exp_scaled.getMaskedImage()
    )

    new_ref = butler_write.put(
        exp_scaled,
        "calexp",
        dataId=ref.dataId,
        run=TEST_CAL_RUN,
    )

    print(
        f"Detector {ref.dataId['detector']}: "
        f"factor={factor:.8f}  "
        f"id={new_ref.id}"
    )

In [ ]:
for dtype, where in [
    (
        "finalVisitSummary",
        "instrument='VIRCAM' AND visit=215044",
    ),
    (
        "skyMap",
        "skymap='lsst_cells_v2'",
    ),
]:
    print(f"\n{dtype}")
    print("-" * len(dtype))

    refs_in = list(
        butler.registry.queryDatasets(
            dtype,
            collections="*",
            where=where,
        )
    )

    for ref in refs_in:
        print(ref.run, ref.dataId)

In [ ]:
TEST_WARP_COLLECTION = "varianceTest/warpScaledK215044"

data_id_warp_test = {
    "instrument": "VIRCAM",
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "visit": 215044,
    "band": "K",
}

warp_scaled = butler.get(
    "direct_warp",
    dataId=data_id_warp_test,
    collections=TEST_WARP_COLLECTION,
)

factors_scaled = scale_calexp_task.computeScaleFactors(
    warp_scaled.getMaskedImage()
)

image_ws = np.asarray(warp_scaled.image.array, dtype=float)
var_ws = np.asarray(warp_scaled.variance.array, dtype=float)
mask_ws = np.asarray(warp_scaled.mask.array)

clean_ws = (
    np.isfinite(image_ws)
    & np.isfinite(var_ws)
    & (var_ws > 0)
    & (mask_ws == 0)
)

z_ws = image_ws[clean_ws] / np.sqrt(var_ws[clean_ws])
med_ws = np.median(z_ws)
nmad_ws = 1.4826 * np.median(
    np.abs(z_ws - med_ws)
)

print("Scaled-calexp → direct_warp test")
print("--------------------------------")
print("Clean pixels          :", clean_ws.sum())
print("Median image/sigma    :", med_ws)
print("NMAD image/sigma      :", nmad_ws)
print("NMAD variance factor  :", nmad_ws**2)
print("Rubin pixelFactor     :", factors_scaled.pixelFactor)
print("Rubin imageFactor     :", factors_scaled.imageFactor)

In [ ]:
STEP3A_RUN = "videoStep3a/20261018T142051Z"

# K direct warps that feed this patch.
warp_refs_k = list(
    butler.registry.queryDatasets(
        "direct_warp",
        collections=STEP3A_RUN,
        where=(
            "instrument='VIRCAM' "
            "AND skymap='lsst_cells_v2' "
            "AND tract=2877 "
            "AND patch=77 "
            "AND band='K'"
        ),
    )
)

visit_ids_k = {
    ref.dataId["visit"]
    for ref in warp_refs_k
}

# All original K calexps, then retain visits used by this patch.
all_calexp_k = list(
    butler.registry.queryDatasets(
        "calexp",
        collections=STEP1_RUN,
        where=(
            "instrument='VIRCAM' "
            "AND band='K'"
        ),
    )
)

calexp_refs_k_patch = [
    ref for ref in all_calexp_k
    if ref.dataId["visit"] in visit_ids_k
]

# Number of detector calexps available for each visit.
from collections import Counter

n_detectors = Counter(
    ref.dataId["visit"]
    for ref in calexp_refs_k_patch
)

print("K-band patch 77 scaling scope")
print("--------------------------------")
print("Direct-warps / visits :", len(visit_ids_k))
print("Calexps               :", len(calexp_refs_k_patch))

print("\nCalexps per visit")
print("  min   :", min(n_detectors.values()))
print("  median:", np.median(list(n_detectors.values())))
print("  max   :", max(n_detectors.values()))

print("\nFirst 10 visits:")
for visit in sorted(visit_ids_k)[:10]:
    print(
        visit,
        "calexps:",
        n_detectors[visit],
    )

### Normalise K-band calexp variance for the test patch

All K-band detector-level `calexp`s contributing to tract 2877, patch 77 are copied to a separate test run and normalised with Rubin's `ScaleVarianceTask`.

The variance correction is measured independently for every detector exposure. No fixed band-level scale factor is used, and the original Step1 datasets remain unchanged.

In [ ]:
TEST_CAL_RUN_K = "varianceTest/calexpScaledK_patch77"

butler_write = Butler(
    REPO,
    writeable=True,
)

# Safe if rerunning the notebook and the run already exists.
try:
    butler_write.registry.registerRun(TEST_CAL_RUN_K)
except Exception:
    pass

factors_k = []

print("Writing scaled K calexps")
print("------------------------")
print("Input calexps:", len(calexp_refs_k_patch))

for i, ref in enumerate(calexp_refs_k_patch, start=1):

    exp_scaled = butler.get(ref).clone()

    factor = scale_calexp_task.run(
        exp_scaled.getMaskedImage()
    )

    butler_write.put(
        exp_scaled,
        "calexp",
        dataId=ref.dataId,
        run=TEST_CAL_RUN_K,
    )

    factors_k.append(
        (
            ref.dataId["visit"],
            ref.dataId["detector"],
            factor,
        )
    )

    if i % 50 == 0 or i == len(calexp_refs_k_patch):
        print(
            f"{i:3d}/{len(calexp_refs_k_patch)} "
            f"calexps written"
        )

factors_arr = np.array(
    [x[2] for x in factors_k],
    dtype=float,
)

print("\nScale-factor summary")
print("--------------------")
print("N      :", len(factors_arr))
print("Median :", np.median(factors_arr))
print("Min    :", np.min(factors_arr))
print("Max    :", np.max(factors_arr))
print(
    "16-84% :",
    np.percentile(factors_arr, [16, 84]),
)

In [ ]:
TEST_WARP_RUN_K = "varianceTest/warpScaledK_patch77"

test_warp_refs = list(
    butler.registry.queryDatasets(
        "direct_warp",
        collections=TEST_WARP_RUN_K,
        where=(
            "instrument='VIRCAM' "
            "AND skymap='lsst_cells_v2' "
            "AND tract=2877 "
            "AND patch=77 "
            "AND band='K'"
        ),
    )
)

print("Number of regenerated K warps:", len(test_warp_refs))

# Spread the sample across the available visits.
sample_indices = np.linspace(
    0,
    len(test_warp_refs) - 1,
    10,
    dtype=int,
)

results = []

print("\nSample variance-normalisation check")
print("-----------------------------------")

for idx in sample_indices:
    ref = test_warp_refs[idx]
    exp = butler.get(ref)

    factors = scale_calexp_task.computeScaleFactors(
        exp.getMaskedImage()
    )

    image_t = np.asarray(exp.image.array, dtype=float)
    var_t = np.asarray(exp.variance.array, dtype=float)
    mask_t = np.asarray(exp.mask.array)

    clean_t = (
        np.isfinite(image_t)
        & np.isfinite(var_t)
        & (var_t > 0)
        & (mask_t == 0)
    )

    z_t = image_t[clean_t] / np.sqrt(var_t[clean_t])
    med_t = np.median(z_t)
    nmad_t = 1.4826 * np.median(
        np.abs(z_t - med_t)
    )

    visit = ref.dataId["visit"]

    results.append(
        (visit, factors.pixelFactor, nmad_t)
    )

    print(
        f"visit={visit:6d}  "
        f"pixelFactor={factors.pixelFactor:8.4f}  "
        f"NMAD={nmad_t:8.4f}"
    )

pixel_factors = np.array([r[1] for r in results])
nmads = np.array([r[2] for r in results])

print("\nSummary")
print("-------")
print(
    "Rubin pixelFactor median/min/max:",
    np.median(pixel_factors),
    np.min(pixel_factors),
    np.max(pixel_factors),
)
print(
    "NMAD median/min/max:",
    np.median(nmads),
    np.min(nmads),
    np.max(nmads),
)

In [ ]:
TEST_STEP3B_K = "varianceTest/step3bScaledK_patch77"

data_id_k = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": "K",
}

# CellCoadd actually used by the useCellCoadds=True measurement path.
cell_new = butler.get(
    "deep_coadd_cell_predetection",
    dataId=data_id_k,
    collections=TEST_STEP3B_K,
)

background_new = butler.get(
    "deep_coadd_background",
    dataId=data_id_k,
    collections=TEST_STEP3B_K,
)

deep_new = butler.get(
    "deep_coadd",
    dataId=data_id_k,
    collections=TEST_STEP3B_K,
)

# Reproduce the Step3c measurement exposure.
stitched_new = cell_new.stitch()
exp_new = stitched_new.asExposure()
exp_new.image.array[:] -= background_new.getImage().array

image_new = np.asarray(exp_new.image.array, dtype=float)
var_new = np.asarray(exp_new.variance.array, dtype=float)
mask_final_new = np.asarray(deep_new.mask.array)

blank_new = (
    np.isfinite(image_new)
    & np.isfinite(var_new)
    & (var_new > 0)
    & (mask_final_new == 0)
)

z_new = image_new[blank_new] / np.sqrt(var_new[blank_new])
med_new = np.median(z_new)
nmad_new = 1.4826 * np.median(np.abs(z_new - med_new))

core_new = np.abs(z_new - med_new) < 5.0 * nmad_new

print("K CellCoadd after calexp variance normalization")
print("------------------------------------------------")
print("Blank-sky pixels       :", blank_new.sum())
print("Median image/sigma     :", med_new)
print("NMAD image/sigma       :", nmad_new)
print("Core STD image/sigma   :", np.std(z_new[core_new]))
print("Implied variance scale :", nmad_new**2)

### End-to-end K-band variance test

The K-band `calexp` variance was normalised independently for all 402 detector exposures contributing to tract 2877, patch 77 using `ScaleVarianceTask`. The correction factors vary between 0.061 and 0.116, with a median of 0.084, demonstrating that a single fixed band-level correction would not be appropriate.

The normalisation is preserved through `MakeDirectWarpTask`: a representative sample of regenerated warps has a median Rubin pixel-based scale factor of 1.024 and a median `image/sqrt(variance)` NMAD of 1.018.

After constructing the CellCoadd from these corrected warps, the K-band blank-sky NMAD improves from approximately 0.30 in the original processing to 1.043, with a core standard deviation of 1.042. The large variance overestimate in the original CellCoadd is therefore primarily inherited from the input `calexp` variance rather than being generated by the CellCoadd assembly itself.

A small residual variance-normalisation difference remains after coaddition, showing that correctly normalised input exposures do not necessarily guarantee an exactly unit-normalised final coadd.

In [ ]:
# Y direct warps contributing to this patch.
warp_refs_y = list(
    butler.registry.queryDatasets(
        "direct_warp",
        collections=STEP3A_RUN,
        where=(
            "instrument='VIRCAM' "
            "AND skymap='lsst_cells_v2' "
            "AND tract=2877 "
            "AND patch=77 "
            "AND band='Y'"
        ),
    )
)

visit_ids_y = {
    ref.dataId["visit"]
    for ref in warp_refs_y
}

# Original Y calexps.
all_calexp_y = list(
    butler.registry.queryDatasets(
        "calexp",
        collections=STEP1_RUN,
        where=(
            "instrument='VIRCAM' "
            "AND band='Y'"
        ),
    )
)

calexp_refs_y_patch = [
    ref for ref in all_calexp_y
    if ref.dataId["visit"] in visit_ids_y
]

from collections import Counter

n_detectors_y = Counter(
    ref.dataId["visit"]
    for ref in calexp_refs_y_patch
)

print("Y-band patch 77 scaling scope")
print("--------------------------------")
print("Direct-warps / visits :", len(visit_ids_y))
print("Calexps               :", len(calexp_refs_y_patch))

print("\nCalexps per visit")
print("  min   :", min(n_detectors_y.values()))
print("  median:", np.median(list(n_detectors_y.values())))
print("  max   :", max(n_detectors_y.values()))

In [ ]:
TEST_CAL_RUN_Y = "varianceTest/calexpScaledY_patch77"

butler_write = Butler(
    REPO,
    writeable=True,
)

try:
    butler_write.registry.registerRun(TEST_CAL_RUN_Y)
except Exception:
    pass

factors_y = []

print("Writing scaled Y calexps")
print("------------------------")
print("Input calexps:", len(calexp_refs_y_patch))

for i, ref in enumerate(calexp_refs_y_patch, start=1):

    exp_scaled = butler.get(ref).clone()

    factor = scale_calexp_task.run(
        exp_scaled.getMaskedImage()
    )

    butler_write.put(
        exp_scaled,
        "calexp",
        dataId=ref.dataId,
        run=TEST_CAL_RUN_Y,
    )

    factors_y.append(
        (
            ref.dataId["visit"],
            ref.dataId["detector"],
            factor,
        )
    )

    if i % 100 == 0 or i == len(calexp_refs_y_patch):
        print(
            f"{i:4d}/{len(calexp_refs_y_patch)} "
            f"calexps written"
        )

factors_y_arr = np.array(
    [x[2] for x in factors_y],
    dtype=float,
)

print("\nY scale-factor summary")
print("----------------------")
print("N      :", len(factors_y_arr))
print("Median :", np.median(factors_y_arr))
print("Min    :", np.min(factors_y_arr))
print("Max    :", np.max(factors_y_arr))
print(
    "16-84% :",
    np.percentile(factors_y_arr, [16, 84]),
)

In [ ]:
TEST_WARP_RUN_Y = "varianceTest/warpScaledY_patch77"

test_warp_refs_y = list(
    butler.registry.queryDatasets(
        "direct_warp",
        collections=TEST_WARP_RUN_Y,
        where=(
            "instrument='VIRCAM' "
            "AND skymap='lsst_cells_v2' "
            "AND tract=2877 "
            "AND patch=77 "
            "AND band='Y'"
        ),
    )
)

print("Number of regenerated Y warps:", len(test_warp_refs_y))

sample_indices_y = np.linspace(
    0,
    len(test_warp_refs_y) - 1,
    10,
    dtype=int,
)

results_y = []

print("\nSample variance-normalisation check")
print("-----------------------------------")

for idx in sample_indices_y:
    ref = test_warp_refs_y[idx]
    exp = butler.get(ref)

    factors = scale_calexp_task.computeScaleFactors(
        exp.getMaskedImage()
    )

    image_t = np.asarray(exp.image.array, dtype=float)
    var_t = np.asarray(exp.variance.array, dtype=float)
    mask_t = np.asarray(exp.mask.array)

    clean_t = (
        np.isfinite(image_t)
        & np.isfinite(var_t)
        & (var_t > 0)
        & (mask_t == 0)
    )

    z_t = image_t[clean_t] / np.sqrt(var_t[clean_t])
    med_t = np.median(z_t)
    nmad_t = 1.4826 * np.median(
        np.abs(z_t - med_t)
    )

    visit = ref.dataId["visit"]

    results_y.append(
        (visit, factors.pixelFactor, nmad_t)
    )

    print(
        f"visit={visit:6d}  "
        f"pixelFactor={factors.pixelFactor:8.4f}  "
        f"NMAD={nmad_t:8.4f}"
    )

pixel_factors_y = np.array([r[1] for r in results_y])
nmads_y = np.array([r[2] for r in results_y])

print("\nSummary")
print("-------")
print(
    "Rubin pixelFactor median/min/max:",
    np.median(pixel_factors_y),
    np.min(pixel_factors_y),
    np.max(pixel_factors_y),
)
print(
    "NMAD median/min/max:",
    np.median(nmads_y),
    np.min(nmads_y),
    np.max(nmads_y),
)

In [ ]:
TEST_STEP3B_Y = "varianceTest/step3bScaledY_patch77"

data_id_y = {
    "skymap": SKYMAP,
    "tract": TRACT,
    "patch": PATCH,
    "band": "Y",
}

# CellCoadd used by the useCellCoadds=True measurement path.
cell_y_new = butler.get(
    "deep_coadd_cell_predetection",
    dataId=data_id_y,
    collections=TEST_STEP3B_Y,
)

background_y_new = butler.get(
    "deep_coadd_background",
    dataId=data_id_y,
    collections=TEST_STEP3B_Y,
)

deep_y_new = butler.get(
    "deep_coadd",
    dataId=data_id_y,
    collections=TEST_STEP3B_Y,
)

# Reproduce the Step3c measurement exposure.
stitched_y_new = cell_y_new.stitch()
exp_y_new = stitched_y_new.asExposure()
exp_y_new.image.array[:] -= background_y_new.getImage().array

image_y_new = np.asarray(exp_y_new.image.array, dtype=float)
var_y_new = np.asarray(exp_y_new.variance.array, dtype=float)
mask_y_final = np.asarray(deep_y_new.mask.array)

blank_y_new = (
    np.isfinite(image_y_new)
    & np.isfinite(var_y_new)
    & (var_y_new > 0)
    & (mask_y_final == 0)
)

z_y_new = (
    image_y_new[blank_y_new]
    / np.sqrt(var_y_new[blank_y_new])
)

med_y_new = np.median(z_y_new)

nmad_y_new = 1.4826 * np.median(
    np.abs(z_y_new - med_y_new)
)

core_y_new = (
    np.abs(z_y_new - med_y_new)
    < 5.0 * nmad_y_new
)

print("Y CellCoadd after calexp variance normalization")
print("------------------------------------------------")
print("Blank-sky pixels       :", blank_y_new.sum())
print("Median image/sigma     :", med_y_new)
print("NMAD image/sigma       :", nmad_y_new)
print("Core STD image/sigma   :", np.std(z_y_new[core_y_new]))
print("Implied variance scale :", nmad_y_new**2)

In [ ]:
# Make a copy of the legacy predetection coadd and put it
# in the same background state as the CellCoadd measurement exposure.
legacy_y_bgsub = legacy_y.clone()

legacy_y_bgsub.image.array[:] -= (
    background_y_new.getImage().array
)

image_legacy_y = np.asarray(
    legacy_y_bgsub.image.array,
    dtype=float,
)

var_legacy_y = np.asarray(
    legacy_y_bgsub.variance.array,
    dtype=float,
)

blank_compare_y = (
    np.isfinite(image_legacy_y)
    & np.isfinite(var_legacy_y)
    & (var_legacy_y > 0)
    & np.isfinite(image_y_new)
    & np.isfinite(var_y_new)
    & (var_y_new > 0)
    & (mask_y_final == 0)
)

med_legacy, nmad_legacy, std_legacy = normalized_scatter(
    image_legacy_y,
    var_legacy_y,
    blank_compare_y,
)

med_cell, nmad_cell, std_cell = normalized_scatter(
    image_y_new,
    var_y_new,
    blank_compare_y,
)

print("Y fair coadd comparison after calexp normalization")
print("---------------------------------------------------")

print("\nLegacy deep_coadd_predetection + same background subtraction")
print("Median image/sigma :", med_legacy)
print("NMAD               :", nmad_legacy)
print("Core STD           :", std_legacy)

print("\nCellCoadd measurement exposure")
print("Median image/sigma :", med_cell)
print("NMAD               :", nmad_cell)
print("Core STD           :", std_cell)

In [ ]:
# Rubin ScaleVariance estimator on the two final Y coadds.
scale_final = ScaleVarianceTask(
    config=assemble_config.scaleWarpVariance
)

legacy_factors_y = scale_final.computeScaleFactors(
    legacy_y_bgsub.getMaskedImage()
)

cell_factors_y = scale_final.computeScaleFactors(
    exp_y_new.getMaskedImage()
)

print("Y final-coadd Rubin variance factors")
print("------------------------------------")

print("\nLegacy deep_coadd_predetection")
print("pixelFactor :", legacy_factors_y.pixelFactor)
print("imageFactor :", legacy_factors_y.imageFactor)

print("\nCellCoadd measurement exposure")
print("pixelFactor :", cell_factors_y.pixelFactor)
print("imageFactor :", cell_factors_y.imageFactor)

In [ ]:
# Final K products from the calexp-normalized test.
cell_k_new = butler.get(
    "deep_coadd_cell_predetection",
    dataId=data_id_k,
    collections=TEST_STEP3B_K,
)

background_k_new = butler.get(
    "deep_coadd_background",
    dataId=data_id_k,
    collections=TEST_STEP3B_K,
)

legacy_k = butler.get(
    "deep_coadd_predetection",
    dataId=data_id_k,
    collections=TEST_STEP3B_K,
)

# Put both into the same background-subtracted state.
legacy_k_bgsub = legacy_k.clone()
legacy_k_bgsub.image.array[:] -= background_k_new.getImage().array

stitched_k = cell_k_new.stitch()
exp_k_cell = stitched_k.asExposure()
exp_k_cell.image.array[:] -= background_k_new.getImage().array

legacy_factors_k = scale_final.computeScaleFactors(
    legacy_k_bgsub.getMaskedImage()
)

cell_factors_k = scale_final.computeScaleFactors(
    exp_k_cell.getMaskedImage()
)

print("K final-coadd Rubin variance factors")
print("------------------------------------")

print("\nLegacy deep_coadd_predetection")
print("pixelFactor :", legacy_factors_k.pixelFactor)
print("imageFactor :", legacy_factors_k.imageFactor)

print("\nCellCoadd measurement exposure")
print("pixelFactor :", cell_factors_k.pixelFactor)
print("imageFactor :", cell_factors_k.imageFactor)

In [ ]:
print("ScaleVariance configuration used for the calexp tests")
print("----------------------------------------------------")

scale_cfg = assemble_config.scaleWarpVariance

cfg = scale_cfg.toDict()

for key, value in cfg.items():
    print(f"{key:35s}: {value}")

In [ ]:
exp_check = butler.get(calexp_refs[0])

mask_check = np.asarray(
    exp_check.mask.array
)

print("Detector:", calexp_refs[0].dataId["detector"])
print("\nMask occupancy")
print("--------------")

for name in exp_check.mask.getMaskPlaneDict():
    bitmask = exp_check.mask.getPlaneBitMask(name)

    frac = np.mean(
        (mask_check & bitmask) != 0
    )

    if frac > 0:
        flag = (
            "IGNORED by ScaleVariance"
            if name in scale_cfg.maskPlanes
            else "NOT ignored"
        )

        print(
            f"{name:20s} "
            f"{100*frac:8.4f}%   "
            f"{flag}"
        )

In [ ]:
import copy

# Extract the real ScaleVarianceConfig from the ConfigurableField proxy.
base_cfg = assemble_config.scaleWarpVariance.value

# Baseline estimator.
baseline_task = ScaleVarianceTask(config=base_cfg)
baseline = baseline_task.computeScaleFactors(
    exp_check.getMaskedImage()
)

# Independent stricter configuration.
strict_cfg = copy.deepcopy(base_cfg)

extra_masks = ["EDGE", "NOT_DEBLENDED"]

strict_cfg.maskPlanes = list(base_cfg.maskPlanes) + extra_masks
strict_cfg.background.ignoredPixelMask = (
    list(base_cfg.background.ignoredPixelMask) + extra_masks
)

strict_task = ScaleVarianceTask(config=strict_cfg)
strict = strict_task.computeScaleFactors(
    exp_check.getMaskedImage()
)

print("K detector 10 ScaleVariance sensitivity to extra masks")
print("--------------------------------------------------------")
print(f"Baseline pixelFactor : {baseline.pixelFactor:.8f}")
print(f"Strict   pixelFactor : {strict.pixelFactor:.8f}")
print(f"Difference           : {100*(strict.pixelFactor/baseline.pixelFactor - 1):+.3f}%")

print()
print(f"Baseline imageFactor : {baseline.imageFactor:.8f}")
print(f"Strict   imageFactor : {strict.imageFactor:.8f}")
print(f"Difference           : {100*(strict.imageFactor/baseline.imageFactor - 1):+.3f}%")

In [ ]:
# Check whether the additional mask planes contribute any NEW pixels
# beyond those already excluded by ScaleVarianceTask.

baseline_mask_bits = exp_check.mask.getPlaneBitMask(
    list(base_cfg.maskPlanes)
)

already_excluded = (mask_check & baseline_mask_bits) != 0

print("Additional mask pixels not already excluded")
print("--------------------------------------------")

for name in ["EDGE", "NOT_DEBLENDED"]:
    bit = exp_check.mask.getPlaneBitMask(name)

    in_plane = (mask_check & bit) != 0
    unique = in_plane & ~already_excluded

    print(
        f"{name:20s} "
        f"total={100*np.mean(in_plane):7.4f}%   "
        f"new={100*np.mean(unique):7.4f}%"
    )

In [ ]:
TEST_STEP3B_YK = "varianceTest/step3bScaledYK_finalScale"

for band, data_id in [("Y", data_id_y), ("K", data_id_k)]:

    metadata = butler.get(
        "deep_coadd.metadata",
        dataId=data_id,
        collections=TEST_STEP3B_YK,
    )

    print(
        f"{band}: VARIANCE_SCALE = "
        f"{metadata.get('VARIANCE_SCALE')}"
    )

In [ ]:
for band, data_id in [("Y", data_id_y), ("K", data_id_k)]:

    # CellCoadd from the new Step3b run.
    cell = butler.get(
        "deep_coadd_cell_predetection",
        dataId=data_id,
        collections=TEST_STEP3B_YK,
    )

    background = butler.get(
        "deep_coadd_background",
        dataId=data_id,
        collections=TEST_STEP3B_YK,
    )

    final_coadd = butler.get(
        "deep_coadd",
        dataId=data_id,
        collections=TEST_STEP3B_YK,
    )

    metadata = butler.get(
        "deep_coadd.metadata",
        dataId=data_id,
        collections=TEST_STEP3B_YK,
    )

    variance_scale = metadata.get("VARIANCE_SCALE")

    # Construct the exposure used for measurement.
    exp = cell.stitch().asExposure()
    exp.image.array[:] -= background.getImage().array

    # Apply the legacy coadd scale factor to the CellCoadd variance in memory.
    exp.variance.array[:] *= variance_scale

    # Rubin estimator after applying the correction.
    after = scale_final.computeScaleFactors(
        exp.getMaskedImage()
    )

    # Independent blank-sky diagnostic using the final detection mask.
    image = np.asarray(exp.image.array, dtype=float)
    variance = np.asarray(exp.variance.array, dtype=float)
    mask = np.asarray(final_coadd.mask.array)

    blank = (
        (mask == 0)
        & np.isfinite(image)
        & np.isfinite(variance)
        & (variance > 0)
    )

    z = image[blank] / np.sqrt(variance[blank])
    med = np.median(z)
    nmad = 1.4826 * np.median(np.abs(z - med))

    core = np.abs(z - med) < 5.0 * nmad
    core_std = np.std(z[core])

    print(f"\n{band}")
    print("-" * 40)
    print(f"Applied VARIANCE_SCALE : {variance_scale:.8f}")
    print(f"Rubin pixelFactor after: {after.pixelFactor:.8f}")
    print(f"Rubin imageFactor after: {after.imageFactor:.8f}")
    print(f"Median image/sigma     : {med:.6f}")
    print(f"Blank-sky NMAD         : {nmad:.6f}")
    print(f"Blank-sky core STD     : {core_std:.6f}")

In [ ]:
def count_by_band(collection, dataset_type):
    refs = list(
        butler.query_datasets(
            dataset_type,
            collections=collection,
            where="instrument='VIRCAM' AND detector IN (10, 11)",
        )
    )

    counts = {}
    for ref in refs:
        band = ref.dataId["band"]
        counts[band] = counts.get(band, 0) + 1

    print(f"\n{collection} : {dataset_type}")
    print("-" * 50)
    print("Total:", len(refs))

    for band in sorted(counts):
        print(f"{band}: {counts[band]}")

count_by_band(
    "videoStep1/20261015T160027Z",
    "calexp",
)

count_by_band(
    "videoStep1ScaleVariance",
    "calexp",
)

In [ ]:
from collections import defaultdict

scaled_refs = list(
    butler.query_datasets(
        "calexp",
        collections="videoStep1ScaleVariance",
        where="instrument='VIRCAM' AND detector IN (10, 11)",
    )
)

print("Scaled calexps found:", len(scaled_refs))

# Completely new dictionary.
all_factors = defaultdict(list)

for ref in scaled_refs:
    # Read only the metadata component rather than the full exposure.
    md = butler.get(
        "calexp.metadata",
        dataId=ref.dataId,
        collections=ref.run,
    )

    factor = md.get("VARIANCE_SCALE")

    if factor is not None:
        all_factors[ref.dataId["band"]].append(float(factor))

print("\nFull calexp variance-scale factors by band")
print("------------------------------------------")

for band in sorted(all_factors):
    values = np.asarray(all_factors[band])

    print(
        f"{band:>2s}  "
        f"N={len(values):4d}  "
        f"median={np.median(values):.4f}  "
        f"16-84%=[{np.percentile(values,16):.4f}, "
        f"{np.percentile(values,84):.4f}]  "
        f"min={np.min(values):.4f}  "
        f"max={np.max(values):.4f}"
    )

In [58]:
import numpy as np

STEP3B_RUNS = {
    "False": "videoStep3bScaleVarianceFalse",
    "True":  "videoStep3bScaleVarianceTrue",
}

bands = ["Z", "Y", "J", "H", "K"]


def normalized_scatter(image, variance, selection):
    z = image[selection] / np.sqrt(variance[selection])

    med = np.median(z)
    nmad = 1.4826 * np.median(np.abs(z - med))

    core = np.abs(z - med) < 5.0 * nmad
    core_std = np.std(z[core])

    return med, nmad, core_std, np.sum(selection)


print("CellCoadd single-pixel normalization")
print("===================================")

results = {}

for band in bands:

    data_id = {
        "skymap": "lsst_cells_v2",
        "tract": 2877,
        "patch": 77,
        "band": band,
    }

    print(f"\n{band}")
    print("-" * 60)

    results[band] = {}

    for label, collection in STEP3B_RUNS.items():

        # Predetection CellCoadd used by the DP2 measurement path.
        cell = butler.get(
            "deep_coadd_cell_predetection",
            dataId=data_id,
            collections=collection,
        )

        # Object background produced by detection.
        background = butler.get(
            "deep_coadd_background",
            dataId=data_id,
            collections=collection,
        )

        # Final legacy coadd supplies the final detection mask.
        final_coadd = butler.get(
            "deep_coadd",
            dataId=data_id,
            collections=collection,
        )

        # Reproduce the exposure state used by Step3c.
        exp = cell.stitch().asExposure()
        exp.image.array[:] -= background.getImage().array

        image = np.asarray(exp.image.array, dtype=float)
        variance = np.asarray(exp.variance.array, dtype=float)
        mask = np.asarray(final_coadd.mask.array)

        blank = (
            (mask == 0)
            & np.isfinite(image)
            & np.isfinite(variance)
            & (variance > 0)
        )

        med, nmad, core_std, n_pix = normalized_scatter(
            image,
            variance,
            blank,
        )

        results[band][label] = {
            "median": med,
            "nmad": nmad,
            "core_std": core_std,
            "n_pix": n_pix,
        }

        print(
            f"{label:5s}  "
            f"N={n_pix:9d}  "
            f"median={med:+.4f}  "
            f"NMAD={nmad:.4f}  "
            f"coreSTD={core_std:.4f}"
        )

CellCoadd single-pixel normalization

Z
------------------------------------------------------------
False  N=  6690289  median=+0.0690  NMAD=1.1108  coreSTD=1.1324
True   N=  6690289  median=+0.0690  NMAD=1.1108  coreSTD=1.1324

Y
------------------------------------------------------------
False  N=  7610545  median=+0.0650  NMAD=1.1015  coreSTD=1.1248
True   N=  7610545  median=+0.0650  NMAD=1.1015  coreSTD=1.1248

J
------------------------------------------------------------
False  N=  8410438  median=+0.0221  NMAD=1.0617  coreSTD=1.0677
True   N=  8410438  median=+0.0221  NMAD=1.0617  coreSTD=1.0677

H
------------------------------------------------------------
False  N=  7142970  median=+0.0148  NMAD=1.0432  coreSTD=1.0429
True   N=  7142970  median=+0.0148  NMAD=1.0432  coreSTD=1.0429

K
------------------------------------------------------------
False  N=  7618948  median=-0.0174  NMAD=1.0423  coreSTD=1.0419
True   N=  7618948  median=-0.0174  NMAD=1.0423  coreSTD=1.0419


In [59]:
from scipy import ndimage
import numpy as np


def fractional_circle(radius, oversample=20):
    half = int(np.ceil(radius)) + 1

    yy, xx = np.mgrid[
        -half:half+1,
        -half:half+1
    ]

    weights = np.zeros_like(xx, dtype=float)

    sub = (
        (np.arange(oversample) + 0.5) / oversample
        - 0.5
    )

    for dy in sub:
        for dx in sub:
            inside = (
                (xx + dx)**2
                + (yy + dy)**2
                <= radius**2
            )
            weights += inside

    weights /= oversample**2

    return weights


def nmad(values):
    med = np.nanmedian(values)
    return 1.4826 * np.nanmedian(
        np.abs(values - med)
    )


RADIUS = 6.0
OVERSAMPLE = 20
N_AP = 2000

weights = fractional_circle(
    RADIUS,
    oversample=OVERSAMPLE,
)

support = weights > 0

rng = np.random.default_rng(12345)

print("DP2 CellCoadd correlated-noise test")
print(f"Radius = {RADIUS} px")
print("===================================")

corr_results = {}

for band in ["Z", "Y", "J", "H", "K"]:

    data_id = {
        "skymap": "lsst_cells_v2",
        "tract": 2877,
        "patch": 77,
        "band": band,
    }

    collection = "videoStep3bScaleVarianceTrue"

    cell = butler.get(
        "deep_coadd_cell_predetection",
        dataId=data_id,
        collections=collection,
    )

    background = butler.get(
        "deep_coadd_background",
        dataId=data_id,
        collections=collection,
    )

    final_coadd = butler.get(
        "deep_coadd",
        dataId=data_id,
        collections=collection,
    )

    exp = cell.stitch().asExposure()
    exp.image.array[:] -= background.getImage().array

    image = np.asarray(
        exp.image.array,
        dtype=float,
    )

    variance = np.asarray(
        exp.variance.array,
        dtype=float,
    )

    mask = np.asarray(
        final_coadd.mask.array
    )

    bad = (
        (mask != 0)
        | ~np.isfinite(image)
        | ~np.isfinite(variance)
        | (variance <= 0)
    )

    image_safe = np.where(
        bad,
        0.0,
        image,
    )

    variance_safe = np.where(
        bad,
        0.0,
        variance,
    )

    # Fractional-aperture flux.
    ap_flux_map = ndimage.convolve(
        image_safe,
        weights,
        mode="constant",
        cval=0.0,
    )

    # Formal variance for the same fractional aperture.
    ap_var_map = ndimage.convolve(
        variance_safe,
        weights**2,
        mode="constant",
        cval=0.0,
    )

    # Require the entire aperture footprint to be clean.
    bad_count = ndimage.convolve(
        bad.astype(np.int16),
        support.astype(np.int16),
        mode="constant",
        cval=1,
    )

    clean_center = (
        (bad_count == 0)
        & np.isfinite(ap_flux_map)
        & np.isfinite(ap_var_map)
        & (ap_var_map > 0)
    )

    candidates = np.flatnonzero(
        clean_center
    )

    if len(candidates) > N_AP:
        chosen = rng.choice(
            candidates,
            size=N_AP,
            replace=False,
        )
    else:
        chosen = candidates

    ap_flux = ap_flux_map.ravel()[chosen]
    ap_sigma = np.sqrt(
        ap_var_map.ravel()[chosen]
    )

    ap_pull = (
        ap_flux - np.median(ap_flux)
    ) / ap_sigma

    correction = nmad(ap_pull)

    corr_results[band] = correction

    print(
        f"{band}: "
        f"N={len(chosen):4d}  "
        f"C(6)={correction:.3f}"
    )

DP2 CellCoadd correlated-noise test
Radius = 6.0 px
Z: N=2000  C(6)=4.130
Y: N=2000  C(6)=3.997
J: N=2000  C(6)=3.230
H: N=2000  C(6)=3.123
K: N=2000  C(6)=2.790


In [60]:
import numpy as np

STEP3C_FALSE = "videoStep3cScaleVarianceFalse"
STEP3C_TRUE  = "videoStep3cScaleVarianceTrue"

dataset_types = [
    "object_unforced_measurement",
    "object_forced_measurement",
]

fields_to_check = [
    "id",
    "base_PsfFlux_instFlux",
    "base_PsfFlux_instFluxErr",
    "base_CircularApertureFlux_6_0_instFlux",
    "base_CircularApertureFlux_6_0_instFluxErr",
]

for dataset_type in dataset_types:

    print(f"\n{dataset_type}")
    print("=" * 70)

    for band in ["Z", "Y", "J", "H", "K"]:

        data_id = {
            "skymap": "lsst_cells_v2",
            "tract": 2877,
            "patch": 77,
            "band": band,
        }

        cat_false = butler.get(
            dataset_type,
            dataId=data_id,
            collections=STEP3C_FALSE,
        )

        cat_true = butler.get(
            dataset_type,
            dataId=data_id,
            collections=STEP3C_TRUE,
        )

        print(f"\n{band}: N(False)={len(cat_false)}, N(True)={len(cat_true)}")

        names_false = set(cat_false.schema.getNames())
        names_true = set(cat_true.schema.getNames())

        for field in fields_to_check:

            if field not in names_false or field not in names_true:
                continue

            a = np.asarray(cat_false[field])
            b = np.asarray(cat_true[field])

            if a.shape != b.shape:
                print(f"  {field}: DIFFERENT SHAPE")
                continue

            if np.issubdtype(a.dtype, np.floating):
                same = np.allclose(
                    a, b,
                    rtol=0.0,
                    atol=0.0,
                    equal_nan=True,
                )

                finite = np.isfinite(a) & np.isfinite(b)

                maxdiff = (
                    np.max(np.abs(a[finite] - b[finite]))
                    if np.any(finite)
                    else np.nan
                )

                print(
                    f"  {field:45s} "
                    f"identical={same}  "
                    f"max|Δ|={maxdiff:.3e}"
                )

            else:
                same = np.array_equal(a, b)

                print(
                    f"  {field:45s} "
                    f"identical={same}"
                )


object_unforced_measurement

Z: N(False)=18048, N(True)=18048
  id                                            identical=True
  base_PsfFlux_instFlux                         identical=False  max|Δ|=1.927e+03
  base_PsfFlux_instFluxErr                      identical=False  max|Δ|=1.706e-01
  base_CircularApertureFlux_6_0_instFlux        identical=False  max|Δ|=2.219e+03
  base_CircularApertureFlux_6_0_instFluxErr     identical=False  max|Δ|=5.488e-02

Y: N(False)=18048, N(True)=18048
  id                                            identical=True
  base_PsfFlux_instFlux                         identical=False  max|Δ|=2.076e+03
  base_PsfFlux_instFluxErr                      identical=False  max|Δ|=4.357e-02
  base_CircularApertureFlux_6_0_instFlux        identical=False  max|Δ|=2.856e+03
  base_CircularApertureFlux_6_0_instFluxErr     identical=False  max|Δ|=5.861e-02

J: N(False)=18048, N(True)=18048
  id                                            identical=True
  base_PsfFlux_instFlux 

This changes one important conclusion: doScaleVariance=True does affect the final Step3c catalogs, even though it does not directly modify the CellCoadd variance.

The reason is that Step3c is not using only the CellCoadd. deconvolve and deblend also receive the legacy deep_coadd. So changing its variance can change the Scarlet/deblending solution, which then changes measured fluxes. The identical IDs and row counts are reassuring—there is no major source-list change—but the flux differences mean we must choose True or False deliberately before CSD3.

In [61]:
import numpy as np

RUN_FALSE = "videoStep3bScaleVarianceFalse"
RUN_TRUE  = "videoStep3bScaleVarianceTrue"

for band in ["Z", "Y", "J", "H", "K"]:

    data_id = {
        "skymap": "lsst_cells_v2",
        "tract": 2877,
        "patch": 77,
        "band": band,
    }

    deep_f = butler.get(
        "deep_coadd",
        dataId=data_id,
        collections=RUN_FALSE,
    )

    deep_t = butler.get(
        "deep_coadd",
        dataId=data_id,
        collections=RUN_TRUE,
    )

    bg_f = butler.get(
        "deep_coadd_background",
        dataId=data_id,
        collections=RUN_FALSE,
    )

    bg_t = butler.get(
        "deep_coadd_background",
        dataId=data_id,
        collections=RUN_TRUE,
    )

    cell_f = butler.get(
        "deep_coadd_cell_predetection",
        dataId=data_id,
        collections=RUN_FALSE,
    ).stitch().asExposure()

    cell_t = butler.get(
        "deep_coadd_cell_predetection",
        dataId=data_id,
        collections=RUN_TRUE,
    ).stitch().asExposure()

    # Legacy deep_coadd
    deep_image_equal = np.array_equal(
        deep_f.image.array,
        deep_t.image.array,
        equal_nan=True,
    )

    deep_mask_equal = np.array_equal(
        deep_f.mask.array,
        deep_t.mask.array,
    )

    good = (
        np.isfinite(deep_f.variance.array)
        & np.isfinite(deep_t.variance.array)
        & (deep_f.variance.array > 0)
        & (deep_t.variance.array > 0)
    )

    var_ratio = np.median(
        deep_t.variance.array[good]
        / deep_f.variance.array[good]
    )

    # Background
    bg_image_f = bg_f.getImage().array
    bg_image_t = bg_t.getImage().array

    bg_equal = np.array_equal(
        bg_image_f,
        bg_image_t,
        equal_nan=True,
    )

    bg_maxdiff = np.nanmax(
        np.abs(bg_image_t - bg_image_f)
    )

    # CellCoadd
    cell_image_equal = np.array_equal(
        cell_f.image.array,
        cell_t.image.array,
        equal_nan=True,
    )

    cell_var_equal = np.array_equal(
        cell_f.variance.array,
        cell_t.variance.array,
        equal_nan=True,
    )

    cell_mask_equal = np.array_equal(
        cell_f.mask.array,
        cell_t.mask.array,
    )

    print(f"\n{band}")
    print("-" * 55)
    print("deep_coadd image identical      :", deep_image_equal)
    print("deep_coadd mask identical       :", deep_mask_equal)
    print("deep_coadd variance ratio T/F   :", var_ratio)
    print("background identical            :", bg_equal)
    print("background max |difference|     :", bg_maxdiff)
    print("CellCoadd image identical       :", cell_image_equal)
    print("CellCoadd variance identical    :", cell_var_equal)
    print("CellCoadd mask identical        :", cell_mask_equal)


Z
-------------------------------------------------------
deep_coadd image identical      : False
deep_coadd mask identical       : True
deep_coadd variance ratio T/F   : 1.3241178
background identical            : False
background max |difference|     : 4.7683716e-07
CellCoadd image identical       : True
CellCoadd variance identical    : True
CellCoadd mask identical        : True

Y
-------------------------------------------------------
deep_coadd image identical      : False
deep_coadd mask identical       : True
deep_coadd variance ratio T/F   : 1.3555967
background identical            : False
background max |difference|     : 8.34465e-07
CellCoadd image identical       : True
CellCoadd variance identical    : True
CellCoadd mask identical        : True

J
-------------------------------------------------------
deep_coadd image identical      : False
deep_coadd mask identical       : True
deep_coadd variance ratio T/F   : 1.2197508
background identical            : False
backgr

In [64]:
import numpy as np

RUN_FALSE = "videoStep3cScaleVarianceFalse"
RUN_TRUE  = "videoStep3cScaleVarianceTrue"

measurements = {
    "PSF": (
        "base_PsfFlux_instFlux",
        "base_PsfFlux_instFluxErr",
    ),
    "Aper6": (
        "base_CircularApertureFlux_6_0_instFlux",
        "base_CircularApertureFlux_6_0_instFluxErr",
    ),
}

print("True - False photometry: object_forced_measurement")
print("====================================================")

for band in ["Z", "Y", "J", "H", "K"]:

    data_id = {
        "skymap": "lsst_cells_v2",
        "tract": 2877,
        "patch": 77,
        "band": band,
    }

    cf = butler.get(
        "object_forced_measurement",
        dataId=data_id,
        collections=RUN_FALSE,
    )

    ct = butler.get(
        "object_forced_measurement",
        dataId=data_id,
        collections=RUN_TRUE,
    )

    print(f"\n{band}")
    print("-" * 72)

    for label, (flux_name, err_name) in measurements.items():

        ff = np.asarray(cf[flux_name], dtype=float)
        ft = np.asarray(ct[flux_name], dtype=float)

        ef = np.asarray(cf[err_name], dtype=float)
        et = np.asarray(ct[err_name], dtype=float)

        good = (
            np.isfinite(ff)
            & np.isfinite(ft)
            & np.isfinite(ef)
            & np.isfinite(et)
            & (ff > 0)
            & (ft > 0)
            & (ef > 0)
            & (et > 0)
            & (ff / ef > 50)
            & (ft / et > 50)
        )

        dm = -2.5 * np.log10(ft[good] / ff[good])

        med = np.median(dm)
        nmad = 1.4826 * np.median(
            np.abs(dm - med)
        )

        p95 = np.percentile(
            np.abs(dm),
            95,
        )

        err_ratio = np.median(
            et[good] / ef[good]
        )

        print(
            f"{label:6s} "
            f"N={np.sum(good):5d}  "
            f"med Δmag={med:+.5f}  "
            f"NMAD={nmad:.5f}  "
            f"P95|Δmag|={p95:.5f}  "
            f"med Err(T/F)={err_ratio:.5f}"
        )

True - False photometry: object_forced_measurement

Z
------------------------------------------------------------------------
PSF    N= 2806  med Δmag=+0.00000  NMAD=0.00000  P95|Δmag|=0.00254  med Err(T/F)=1.00000
Aper6  N= 2550  med Δmag=+0.00000  NMAD=0.00000  P95|Δmag|=0.00249  med Err(T/F)=1.00000

Y
------------------------------------------------------------------------
PSF    N= 2903  med Δmag=-0.00000  NMAD=0.00000  P95|Δmag|=0.00127  med Err(T/F)=1.00000
Aper6  N= 2533  med Δmag=+0.00000  NMAD=0.00000  P95|Δmag|=0.00144  med Err(T/F)=1.00000

J
------------------------------------------------------------------------
PSF    N= 2113  med Δmag=-0.00000  NMAD=0.00000  P95|Δmag|=0.00101  med Err(T/F)=1.00000
Aper6  N= 1825  med Δmag=+0.00000  NMAD=0.00000  P95|Δmag|=0.00134  med Err(T/F)=1.00000

H
------------------------------------------------------------------------
PSF    N= 1395  med Δmag=-0.00000  NMAD=0.00000  P95|Δmag|=0.00095  med Err(T/F)=1.00000
Aper6  N= 1208  med Δm

- The new calexp ScaleVariance task fixes the major DP2 variance-normalisation problem;
- doScaleVariance=True does not change the CellCoadd itself;
- It does influence Step3c indirectly through the legacy coadd, but the impact on high-S/N photometry is tiny;
- Therefore I would keep doScaleVariance=True so the legacy deep_coadd is also correctly normalised.

### Reproducibility note

This notebook records the development and validation of the VIRCAM variance-normalisation changes for the DP2 ELAIS-S1 test processing. The analysis was performed using a local Butler repository containing the intermediate Step1–Step3 products for tract 2877, patch 77. Because the full Butler repository is very large and is not retained permanently, some cells may not be directly reproducible from this notebook alone. The notebook is therefore intended primarily as a record of the tests, diagnostics, and decisions used to validate the variance treatment.